# Factor Analysis

This notebook combines systematic factor attribution with peer-relative price analysis. It measures rolling factor exposures, alpha, explanatory power, and idiosyncratic risk alongside GICS peer performance, correlations, and market-cap-weighted industry benchmarks.

In [ ]:
# 1. Setup
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import display

# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()
SINGLE_ASSET_DIRECTORY = PROJECT_ROOT / "Research" / "Single Asset"
if str(SINGLE_ASSET_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SINGLE_ASSET_DIRECTORY))
from _params import get_single_asset_params

from Quantapp.data import (
    GICSDataClient,
    build_capitalization_count_table,
    build_gics_peer_frames,
    build_gics_peer_table,
    choose_gics_peer_level,
    get_market_history,
    normalize_peer_symbol,
    select_gics_peer_rows,
)
from Quantapp.data.sources import (
    fetch_fmp_historical_market_capitalization,
    normalize_fmp_symbol,
)
from Quantapp.models import FactorRegressionModel
from Quantapp.visualization.core import configure_plotly_notebook_renderers
from Quantapp.visualization.views.single_asset_profile.pricing.factor_analysis import (
    plot_idiosyncratic_risk_view,
    plot_rolling_regression_view,
)

configure_plotly_notebook_renderers()


In [ ]:
# 2. Shared and notebook-specific parameters
pricing_params = get_single_asset_params()
ticker_str = pricing_params["ticker_str"]
period = pricing_params["period"]
interval = pricing_params["interval"]
price_field = "Close"
sector_benchmark_symbol = "SOXX"
pca_years = None  # None = use all available price history.
plot_years = None  # None = plot all available index history.
market_cap_history_limit = 5000  # FMP historical market-cap endpoint max history per symbol.
refresh_market_cap_cache = False
requery_incomplete_market_cap_cache = True  # One retry for short cache files, then request log prevents repeat retries.
market_cap_request_pause_seconds = 0.25  # About 240 calls/minute, below the 300/min FMP limit.
market_cap_cache_dir = PROJECT_ROOT / "company_data" / "fmp_historical_market_caps"
peer_index_cache_dir = PROJECT_ROOT / "company_data" / "factor_peer_indexes"
gics_company_cache_path = PROJECT_ROOT / "company_data" / "gics_companies.csv"
refresh_gics_company_cache = False
show_non_market_cap_indexes = False  # Keep PCA/equal-weight/vol-adjusted code tucked away unless needed.
factor_proxy_symbols = ["SPY", "SIZE", "VLUE", "QUAL", "USMV", "MTUM", "BIL"]

peer_group_level = "Sub-Industry"
peer_limit = 12
minimum_peer_count = 4
peer_capitalizations = None  # None = include Large / Mid / Small Cap; use "same_as_target" for one cap bucket.
show_full_peer_tables = True

return_windows = {
    "1M": 21,
    "3M": 63,
    "6M": 126,
    "1Y": 252,
}
rolling_window = 63
factor_rolling_window = 252
factor_verbose = True

ticker_str


In [ ]:
# 3. Build GICS peer hierarchy
gics_data = GICSDataClient(save_path=PROJECT_ROOT)
if gics_company_cache_path.exists() and not refresh_gics_company_cache:
    all_companies = pd.read_csv(gics_company_cache_path)
    print(f"Loaded {len(all_companies):,} companies from the local GICS cache.")
else:
    all_companies = gics_data.retrieve_companies()
    gics_company_cache_path.parent.mkdir(parents=True, exist_ok=True)
    all_companies.to_csv(gics_company_cache_path, index=False)
    print(f"Refreshed the local GICS cache with {len(all_companies):,} companies.")

gics_peer_context = build_gics_peer_frames(
    ticker_str,
    companies=all_companies,
    capitalizations=peer_capitalizations,
    symbol_label="YFinance Symbol",
)

target_symbol = gics_peer_context.target_symbol
target_gics_row = gics_peer_context.target_row
peer_summary = gics_peer_context.summary

sector_peer_universe = gics_peer_context.frame("Sector")
industry_group_peer_universe = gics_peer_context.frame("Industry Group")
industry_peer_universe = gics_peer_context.frame("Industry")
sub_industry_peer_universe = gics_peer_context.frame("Sub-Industry")

sector_peer_table = gics_peer_context.table("Sector")
industry_group_peer_table = gics_peer_context.table("Industry Group")
industry_peer_table = gics_peer_context.table("Industry")
sub_industry_peer_table = gics_peer_context.table("Sub-Industry")

peer_group_level_used, selected_gics_peer_universe = choose_gics_peer_level(
    gics_peer_context,
    preferred_level=peer_group_level,
    minimum_peer_count=minimum_peer_count,
)
peer_group_value = target_gics_row[peer_group_level_used]
selected_peer_universe = select_gics_peer_rows(
    selected_gics_peer_universe,
    peer_count=peer_limit,
    target_capitalization=target_gics_row.get("Capitalization"),
)
selected_peer_table = build_gics_peer_table(selected_peer_universe, symbol_label="YFinance Symbol")
selected_peer_capitalization_counts = build_capitalization_count_table(selected_peer_table)
selected_peer_symbols = selected_peer_universe["Normalized Symbol"].tolist()

print(f"GICS peer hierarchy for {target_symbol}")
print(f"Capitalization filter: {gics_peer_context.capitalization_filter}")
display(peer_summary)

print(f"Sector: {target_gics_row['Sector']} ({len(sector_peer_table)} peers)")
display(sector_peer_table)

gics_peer_treemap_levels = ["Root", "Industry Group", "Industry", "Sub-Industry"]
target_treemap_row = target_gics_row.to_frame().T.copy()
target_treemap_row["Normalized Symbol"] = target_symbol
gics_peer_treemap_source = pd.concat(
    [target_treemap_row, sector_peer_universe], ignore_index=True
).drop_duplicates("Normalized Symbol")

if gics_peer_treemap_source.empty:
    print("No sector peers available for the GICS peer treemap.")
else:
    def clean_gics_treemap_value(value):
        return "Unclassified" if pd.isna(value) else str(value)

    gics_peer_treemap_source["Root"] = clean_gics_treemap_value(target_gics_row["Sector"])
    gics_peer_treemap_source["Company Count"] = 1
    gics_peer_treemap_source[gics_peer_treemap_levels + ["Normalized Symbol"]] = (
        gics_peer_treemap_source[gics_peer_treemap_levels + ["Normalized Symbol"]]
        .fillna("Unclassified")
        .astype(str)
    )
    gics_peer_treemap_source["Capitalization"] = (
        gics_peer_treemap_source["Capitalization"].fillna("Unclassified").astype(str)
    )
    target_treemap_path_values = tuple(
        clean_gics_treemap_value(target_gics_row[column])
        for column in ["Sector", "Industry Group", "Industry", "Sub-Industry"]
    )
    target_peer_container_tree_level = (
        "Root" if peer_group_level_used == "Sector" else peer_group_level_used
    )
    target_peer_container_level_index = gics_peer_treemap_levels.index(
        target_peer_container_tree_level
    )
    target_peer_container_path_values = target_treemap_path_values[
        : target_peer_container_level_index + 1
    ]

    import colorsys

    target_container_border_color = "#22d3ee"
    target_company_border_color = "#f43f5e"
    default_border_color = "#111827"
    gics_level_display_names = {"Root": "Sector"}
    gics_company_capitalization_colors = {
        "Large Cap": "#2563eb",
        "Mid Cap": "#16a34a",
        "Small Cap": "#f97316",
        "Unclassified": "#9ca3af",
    }
    gics_sector_hue_anchors = {
        "Communication Services": 276,
        "Consumer Discretionary": 24,
        "Consumer Staples": 96,
        "Energy": 44,
        "Financials": 208,
        "Health Care": 344,
        "Industrials": 184,
        "Information Technology": 242,
        "Materials": 136,
        "Real Estate": 304,
        "Utilities": 58,
        "Unclassified": 214,
    }
    gics_hierarchy_level_styles = {
        0: {"saturation": 0.48, "lightness": 0.26},  # Sector
        1: {"saturation": 0.58, "lightness": 0.36},  # Industry Group
        2: {"saturation": 0.66, "lightness": 0.48},  # Industry
        3: {"saturation": 0.72, "lightness": 0.60},  # Sub-Industry
        4: {"saturation": 0.66, "lightness": 0.72},  # Company leaf
    }
    gics_hierarchy_hue_spread = {1: 46, 2: 24, 3: 12}

    def stable_color_number(value):
        return sum((position + 1) * ord(character) for position, character in enumerate(str(value)))

    def hsl_to_hex(hue_degrees, saturation, lightness):
        red, green, blue = colorsys.hls_to_rgb((hue_degrees % 360) / 360, lightness, saturation)
        return f"#{int(red * 255):02x}{int(green * 255):02x}{int(blue * 255):02x}"

    def text_color_for_background(hex_color):
        hex_color = hex_color.lstrip("#")
        red, green, blue = [int(hex_color[index : index + 2], 16) / 255 for index in (0, 2, 4)]

        def linearize(channel):
            return channel / 12.92 if channel <= 0.03928 else ((channel + 0.055) / 1.055) ** 2.4

        luminance = 0.2126 * linearize(red) + 0.7152 * linearize(green) + 0.0722 * linearize(blue)
        return "#111827" if luminance > 0.46 else "#f8fafc"

    gics_hierarchy_sibling_index = {}
    for level_index, level_name in enumerate(gics_peer_treemap_levels):
        path_columns = gics_peer_treemap_levels[: level_index + 1]
        if level_index == 0:
            sibling_values = sorted(gics_peer_treemap_source[level_name].dropna().astype(str).unique())
            for sibling_position, sibling_value in enumerate(sibling_values):
                gics_hierarchy_sibling_index[(level_index, (sibling_value,))] = (
                    sibling_position,
                    len(sibling_values),
                )
            continue

        parent_columns = path_columns[:-1]
        for parent_values, group in gics_peer_treemap_source.groupby(parent_columns, sort=True):
            if not isinstance(parent_values, tuple):
                parent_values = (parent_values,)
            parent_values = tuple(map(str, parent_values))
            sibling_values = sorted(group[level_name].dropna().astype(str).unique())
            for sibling_position, sibling_value in enumerate(sibling_values):
                gics_hierarchy_sibling_index[(level_index, parent_values + (sibling_value,))] = (
                    sibling_position,
                    len(sibling_values),
                )

    def centered_sibling_hue_offset(level_index, path_values):
        sibling_position, sibling_count = gics_hierarchy_sibling_index.get(
            (level_index, tuple(path_values)),
            (0, 1),
        )
        if sibling_count <= 1:
            return 0
        spread = gics_hierarchy_hue_spread.get(level_index, 0)
        return ((sibling_position / (sibling_count - 1)) - 0.5) * 2 * spread

    def stable_centered_hue_offset(value, spread):
        return (stable_color_number(value) % (spread * 2 + 1)) - spread

    def hierarchy_color_for_path(path_values, level_index, company_symbol=None):
        path_values = tuple(map(str, path_values))
        base_hue = gics_sector_hue_anchors.get(path_values[0], stable_color_number(path_values[0]) % 360)
        hue = base_hue
        for path_level_index in range(1, min(level_index, len(path_values) - 1) + 1):
            hue += centered_sibling_hue_offset(path_level_index, path_values[: path_level_index + 1])
        if company_symbol is not None:
            hue += stable_centered_hue_offset(company_symbol, 4)
        style = gics_hierarchy_level_styles.get(level_index, gics_hierarchy_level_styles[4])
        return hsl_to_hex(hue, style["saturation"], style["lightness"])

    def format_gics_path(path_columns, path_values):
        return "<br>".join(
            f"{gics_level_display_names.get(column, column)}: {value}"
            for column, value in zip(path_columns, path_values)
        )

    def summarize_capitalization(group):
        counts = group["Capitalization"].value_counts()
        ordered_buckets = ["Large Cap", "Mid Cap", "Small Cap", "Unclassified"]
        lines = [
            f"{bucket}: {int(counts[bucket])}"
            for bucket in ordered_buckets
            if bucket in counts.index
        ]
        if not lines:
            return "Capitalization: Unclassified"
        return "Capitalization:<br>" + "<br>".join(lines)

    selected_peer_symbol_set = set(selected_peer_symbols)
    gics_peer_treemap_rows = []
    for level_index, level_name in enumerate(gics_peer_treemap_levels):
        path_columns = gics_peer_treemap_levels[: level_index + 1]
        for path_values, group in gics_peer_treemap_source.groupby(path_columns, sort=True):
            if not isinstance(path_values, tuple):
                path_values = (path_values,)
            path_values = tuple(map(str, path_values))
            is_target_container = (
                level_name == target_peer_container_tree_level
                and path_values == target_peer_container_path_values
            )
            cap_text = summarize_capitalization(group)
            node_color = hierarchy_color_for_path(path_values, level_index)
            target_container_text = "<br>Direct Peer Container: Yes" if is_target_container else ""
            gics_peer_treemap_rows.append(
                {
                    "Node ID": " > ".join(path_values),
                    "Parent ID": " > ".join(path_values[:-1]) if level_index else "",
                    "Label": path_values[-1],
                    "GICS Level": gics_level_display_names.get(level_name, level_name),
                    "Path Text": format_gics_path(path_columns, path_values),
                    "Capitalization Text": cap_text + target_container_text,
                    "Company Count": int(group["Normalized Symbol"].nunique()),
                    "Color": node_color,
                    "Display Text": f"{path_values[-1]}<br>{int(group['Normalized Symbol'].nunique())}",
                    "Border Color": target_container_border_color if is_target_container else default_border_color,
                    "Border Width": 5 if is_target_container else 1,
                    "Text Color": text_color_for_background(node_color),
                }
            )

    company_rows = gics_peer_treemap_source.drop_duplicates("Normalized Symbol")
    for _, row in company_rows.iterrows():
        parent_values = tuple(row[column] for column in gics_peer_treemap_levels)
        parent_id = " > ".join(parent_values)
        symbol = row["Normalized Symbol"]
        cap = row["Capitalization"]
        company_color = gics_company_capitalization_colors.get(cap, "#9ca3af")
        is_target_company = symbol == target_symbol
        is_selected_peer = symbol in selected_peer_symbol_set
        company_role = "Target Company" if is_target_company else "Selected Peer" if is_selected_peer else "Sector Peer"
        gics_peer_treemap_rows.append(
            {
                "Node ID": f"{parent_id} > {symbol}",
                "Parent ID": parent_id,
                "Label": symbol,
                "GICS Level": "Company",
                "Path Text": format_gics_path(gics_peer_treemap_levels, parent_values)
                + f"<br>Ticker: {symbol}",
                "Capitalization Text": f"Capitalization: {cap}<br>Role: {company_role}",
                "Company Count": 1,
                "Color": company_color,
                "Display Text": f"{symbol}<br>Target" if is_target_company else symbol,
                "Border Color": target_company_border_color if is_target_company else default_border_color,
                "Border Width": 6 if is_target_company else 1,
                "Text Color": text_color_for_background(company_color),
            }
        )

    gics_peer_treemap_nodes = pd.DataFrame(gics_peer_treemap_rows)


In [ ]:
# 4. Fetch price history for target and GICS peer groups
peer_group_frames = {
    "Sector": sector_peer_universe,
    "Industry Group": industry_group_peer_universe,
    "Industry": industry_peer_universe,
    "Sub-Industry": sub_industry_peer_universe,
    "Selected Peers": selected_peer_universe,
}

requested_symbols = [target_symbol, sector_benchmark_symbol, *factor_proxy_symbols]
for frame in peer_group_frames.values():
    requested_symbols.extend(frame["Normalized Symbol"].tolist())
requested_symbols = [symbol for symbol in dict.fromkeys(requested_symbols) if symbol]

market_histories = get_market_history(
    symbols=requested_symbols,
    period=period,
    interval=interval,
    provider="yfinance",
    align=False,
)

def extract_price_series(history_map, symbol, field="Close"):
    frame = history_map.get(symbol, pd.DataFrame())
    if frame is None or frame.empty:
        return pd.Series(dtype="float64", name=symbol)
    if field in frame.columns:
        series = frame[field]
    elif "Close" in frame.columns:
        series = frame["Close"]
    else:
        return pd.Series(dtype="float64", name=symbol)
    return pd.to_numeric(series, errors="coerce").rename(symbol).dropna()

price_series = [
    extract_price_series(market_histories, symbol, field=price_field)
    for symbol in requested_symbols
]
price_series = [series for series in price_series if not series.empty]
price_frame = pd.concat(price_series, axis=1).sort_index().ffill()
available_symbols = set(price_frame.columns)

missing_symbols = [symbol for symbol in requested_symbols if symbol not in available_symbols]
print(f"Loaded {len(available_symbols)} of {len(requested_symbols)} requested symbols.")
if not price_frame.empty:
    print(f"Price window loaded: {price_frame.index.min():%Y-%m-%d} to {price_frame.index.max():%Y-%m-%d}")
if missing_symbols:
    print(f"Missing symbols: {missing_symbols[:20]}{'...' if len(missing_symbols) > 20 else ''}")

display(price_frame.tail())


## Factor attribution and idiosyncratic risk

This section complements the peer analysis by separating the asset's systematic factor exposures from residual, asset-specific risk.

In [ ]:
# Run rolling factor attribution and idiosyncratic-risk analysis
# Reuse the batched price query from block 4 instead of downloading the asset twice.
missing_factor_symbols = [symbol for symbol in factor_proxy_symbols if symbol not in price_frame.columns]
if missing_factor_symbols:
    raise ValueError(f"Missing factor proxy histories: {missing_factor_symbols}")
if target_symbol not in price_frame.columns:
    raise ValueError(f"Missing target price history for {target_symbol}.")

factor_model = FactorRegressionModel()
factor_proxy_prices = price_frame[factor_proxy_symbols].dropna(how="all")
factor_proxy_returns = factor_proxy_prices.pct_change(fill_method=None).dropna()
factor_sets = factor_model.build_ff5_proxy_factor_returns(factor_proxy_returns)
stock_returns = price_frame[target_symbol].pct_change(fill_method=None).dropna()
rolling_factor_results = factor_model.rolling_factor_regression(
    stock_returns=stock_returns,
    rf_series=factor_proxy_returns["BIL"],
    factor_returns=factor_sets["ff5"],
    window=factor_rolling_window,
    auto_window=True,
    verbose=factor_verbose,
)
factor_returns = factor_sets["all"]
factor_returns_capm = factor_sets["capm"]
factor_returns_ff3 = factor_sets["ff3"]
factor_returns_ff5 = factor_sets["ff5"]

factor_figures = plot_rolling_regression_view(
    rolling_factor_results,
    ticker_str,
    factor_returns_ff5,
)
for figure_name in ("alpha", "betas", "r_squared"):
    factor_figures[figure_name].show(config={"responsive": True, "scrollZoom": True})

idiosyncratic_risk_figure = plot_idiosyncratic_risk_view(
    rolling_factor_results,
    ticker_str,
    template="plotly_dark",
)
idiosyncratic_risk_figure.show(config={"responsive": True, "scrollZoom": True})


In [ ]:
# 5. Build market-cap weighted sector indexes
def build_pca_index(price_data, symbols, index_name="Sector PCA Index", base_value=100.0, min_history_ratio=0.80):
    available = [symbol for symbol in dict.fromkeys(symbols) if symbol in price_data.columns]
    if len(available) < 2:
        raise ValueError(f"PCA index needs at least 2 available symbols; found {len(available)}.")

    candidate_prices = price_data[available].copy()
    min_observations = max(2, int(np.ceil(len(candidate_prices) * min_history_ratio)))
    candidate_prices = candidate_prices.dropna(axis=1, thresh=min_observations).ffill().dropna()
    candidate_prices = candidate_prices.loc[:, candidate_prices.gt(0).all()]
    if candidate_prices.shape[1] < 2:
        raise ValueError(
            f"PCA index needs at least 2 symbols with usable price history; found {candidate_prices.shape[1]}."
        )

    log_returns = np.log(candidate_prices).diff().replace([np.inf, -np.inf], np.nan).dropna()
    return_means = log_returns.mean()
    return_stds = log_returns.std(ddof=0).replace(0, np.nan)
    standardized_returns = ((log_returns - return_means) / return_stds).dropna(axis=1)
    log_returns = log_returns[standardized_returns.columns]
    if standardized_returns.shape[1] < 2:
        raise ValueError(
            f"PCA index needs at least 2 non-constant return series; found {standardized_returns.shape[1]}."
        )

    _, singular_values, components = np.linalg.svd(standardized_returns.to_numpy(), full_matrices=False)
    loadings = pd.Series(components[0], index=standardized_returns.columns, name="PC1 Loading")
    if loadings.sum() < 0:
        loadings = -loadings

    positive_loadings = loadings.clip(lower=0)
    if positive_loadings.sum() == 0:
        index_weights = loadings.abs() / loadings.abs().sum()
    else:
        index_weights = positive_loadings / positive_loadings.sum()
    index_weights.name = "PCA Index Weight"

    index_log_returns = log_returns[index_weights.index].mul(index_weights, axis=1).sum(axis=1)
    pca_values = (base_value * np.exp(index_log_returns.cumsum())).rename(index_name)
    pca_index = pd.concat(
        [pd.Series([base_value], index=candidate_prices.index[:1], name=index_name), pca_values]
    )
    explained_variance = singular_values ** 2
    explained_variance_ratio = float(explained_variance[0] / explained_variance.sum())
    loading_table = pd.concat([loadings, index_weights], axis=1).sort_values(
        "PCA Index Weight", ascending=False
    )
    return pca_index, loading_table, explained_variance_ratio, log_returns

def build_volatility_adjusted_equal_weight_index(price_data, symbols, index_name, base_value=100.0, min_history_ratio=0.80):
    available = [symbol for symbol in dict.fromkeys(symbols) if symbol in price_data.columns]
    if len(available) < 2:
        raise ValueError(f"Volatility-adjusted index needs at least 2 available symbols; found {len(available)}.")

    candidate_prices = price_data[available].copy()
    min_observations = max(2, int(np.ceil(len(candidate_prices) * min_history_ratio)))
    candidate_prices = candidate_prices.dropna(axis=1, thresh=min_observations).ffill().dropna()
    candidate_prices = candidate_prices.loc[:, candidate_prices.gt(0).all()]
    if candidate_prices.shape[1] < 2:
        raise ValueError(
            f"Volatility-adjusted index needs at least 2 symbols with usable price history; found {candidate_prices.shape[1]}."
        )

    log_returns = np.log(candidate_prices).diff().replace([np.inf, -np.inf], np.nan).dropna()
    realized_volatility = log_returns.std(ddof=0).replace(0, np.nan).dropna()
    inverse_volatility = (1 / realized_volatility).replace([np.inf, -np.inf], np.nan).dropna()
    if len(inverse_volatility) < 2:
        raise ValueError(f"Volatility-adjusted index needs at least 2 non-constant return series; found {len(inverse_volatility)}.")

    index_weights = (inverse_volatility / inverse_volatility.sum()).rename("Volatility-Adjusted Weight")
    index_log_returns = log_returns[index_weights.index].mul(index_weights, axis=1).sum(axis=1)
    index_values = (base_value * np.exp(index_log_returns.cumsum())).rename(index_name)
    volatility_adjusted_index = pd.concat(
        [pd.Series([base_value], index=candidate_prices.index[:1], name=index_name), index_values]
    )
    weight_table = pd.concat(
        [(realized_volatility[index_weights.index] * np.sqrt(252)).rename("Annualized Volatility"), index_weights],
        axis=1,
    ).sort_values("Volatility-Adjusted Weight", ascending=False)
    return volatility_adjusted_index, weight_table, index_log_returns.rename(f"{index_name} Log Return")

def build_equal_weight_index(price_data, symbols, index_name, base_value=100.0, min_history_ratio=0.80):
    available = [symbol for symbol in dict.fromkeys(symbols) if symbol in price_data.columns]
    if len(available) < 2:
        raise ValueError(f"Equal-weight index needs at least 2 available symbols; found {len(available)}.")

    candidate_prices = price_data[available].copy()
    min_observations = max(2, int(np.ceil(len(candidate_prices) * min_history_ratio)))
    candidate_prices = candidate_prices.dropna(axis=1, thresh=min_observations).ffill().dropna()
    candidate_prices = candidate_prices.loc[:, candidate_prices.gt(0).all()]
    if candidate_prices.shape[1] < 2:
        raise ValueError(
            f"Equal-weight index needs at least 2 symbols with usable price history; found {candidate_prices.shape[1]}."
        )

    log_returns = np.log(candidate_prices).diff().replace([np.inf, -np.inf], np.nan).dropna()
    index_weights = pd.Series(
        1 / log_returns.shape[1],
        index=log_returns.columns,
        name="Equal Weight",
    )
    index_log_returns = log_returns.mul(index_weights, axis=1).sum(axis=1)
    index_values = (base_value * np.exp(index_log_returns.cumsum())).rename(index_name)
    equal_weight_index = pd.concat(
        [pd.Series([base_value], index=candidate_prices.index[:1], name=index_name), index_values]
    )
    weight_table = index_weights.to_frame().sort_values("Equal Weight", ascending=False)
    return equal_weight_index, weight_table, index_log_returns.rename(f"{index_name} Log Return")


def _market_cap_cache_path(cache_dir, symbol):
    safe_symbol = normalize_fmp_symbol(symbol).replace(".", "_").replace(" ", "")
    return Path(cache_dir) / f"{safe_symbol}.csv"

def _market_cap_series_from_frame(frame, symbol):
    if frame is None or frame.empty:
        return pd.Series(dtype="float64", name=symbol)

    normalized_columns = {str(column).strip().lower(): column for column in frame.columns}
    date_column = normalized_columns.get("date") or normalized_columns.get("dates")
    market_cap_column = (
        normalized_columns.get("marketcap")
        or normalized_columns.get("market_cap")
        or normalized_columns.get("market cap")
    )
    if date_column is None or market_cap_column is None:
        return pd.Series(dtype="float64", name=symbol)

    cleaned = frame[[date_column, market_cap_column]].copy()
    cleaned[date_column] = pd.to_datetime(cleaned[date_column], errors="coerce").dt.tz_localize(None)
    cleaned[market_cap_column] = pd.to_numeric(cleaned[market_cap_column], errors="coerce")
    cleaned = cleaned.dropna(subset=[date_column, market_cap_column])
    cleaned = cleaned.loc[cleaned[market_cap_column] > 0]
    cleaned = cleaned.drop_duplicates(subset=[date_column], keep="last").sort_values(date_column)
    if cleaned.empty:
        return pd.Series(dtype="float64", name=symbol)
    return cleaned.set_index(date_column)[market_cap_column].rename(symbol)

_market_cap_request_log_cache = {}

def _market_cap_request_log_path(cache_dir):
    return Path(cache_dir) / "_request_log.csv"

def _load_market_cap_request_log(cache_dir):
    log_path = _market_cap_request_log_path(cache_dir)
    cache_key = str(log_path.resolve())
    if cache_key in _market_cap_request_log_cache:
        return _market_cap_request_log_cache[cache_key]
    columns = [
        "Symbol",
        "Requested From",
        "Requested To",
        "Limit",
        "Fetched At",
        "Rows",
        "Data Start",
        "Data End",
    ]
    if not log_path.exists():
        log_frame = pd.DataFrame(columns=columns)
        _market_cap_request_log_cache[cache_key] = log_frame
        return log_frame
    log_frame = pd.read_csv(log_path)
    for column in columns:
        if column not in log_frame.columns:
            log_frame[column] = pd.NA
    log_frame = log_frame[columns]
    _market_cap_request_log_cache[cache_key] = log_frame
    return log_frame

def _normalize_market_cap_limit_key(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return ""
    try:
        return str(int(float(value)))
    except (TypeError, ValueError):
        return str(value).strip()

def _append_market_cap_request_log(cache_dir, symbol, limit, from_date, to_date, market_cap_series):
    log_path = _market_cap_request_log_path(cache_dir)
    log_frame = _load_market_cap_request_log(cache_dir)
    if market_cap_series.empty:
        data_start = pd.NA
        data_end = pd.NA
    else:
        data_start = market_cap_series.index.min().strftime("%Y-%m-%d")
        data_end = market_cap_series.index.max().strftime("%Y-%m-%d")
    row = {
        "Symbol": normalize_fmp_symbol(symbol),
        "Requested From": from_date or "",
        "Requested To": to_date or "",
        "Limit": _normalize_market_cap_limit_key(limit),
        "Fetched At": pd.Timestamp.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),
        "Rows": int(len(market_cap_series)),
        "Data Start": data_start,
        "Data End": data_end,
    }
    log_frame = pd.concat([log_frame, pd.DataFrame([row])], ignore_index=True)
    log_frame.to_csv(log_path, index=False)
    _market_cap_request_log_cache[str(log_path.resolve())] = log_frame

def _market_cap_request_was_attempted(cache_dir, symbol, limit, from_date, to_date):
    log_frame = _load_market_cap_request_log(cache_dir)
    if log_frame.empty:
        return False

    symbol_key = normalize_fmp_symbol(symbol)
    requested_start = pd.Timestamp(from_date) if from_date else None
    requested_end = pd.Timestamp(to_date) if to_date else None
    limit_key = _normalize_market_cap_limit_key(limit)

    symbol_log = log_frame.loc[log_frame["Symbol"].astype(str).eq(symbol_key)].copy()
    symbol_log = symbol_log.loc[symbol_log["Limit"].map(_normalize_market_cap_limit_key).eq(limit_key)]
    if symbol_log.empty:
        return False

    for _, row in symbol_log.iterrows():
        row_start = pd.to_datetime(row.get("Requested From"), errors="coerce")
        row_end = pd.to_datetime(row.get("Requested To"), errors="coerce")
        covers_start = requested_start is None or pd.isna(row_start) or row_start <= requested_start
        covers_end = requested_end is None or pd.isna(row_end) or row_end >= requested_end
        if covers_start and covers_end:
            return True
    return False

def _respect_market_cap_request_pause(pause_seconds):
    if not pause_seconds or pause_seconds <= 0:
        return
    last_request_at = globals().get("_last_fmp_market_cap_request_at")
    now = time.monotonic()
    if last_request_at is not None:
        elapsed = now - last_request_at
        if elapsed < pause_seconds:
            time.sleep(pause_seconds - elapsed)
    globals()["_last_fmp_market_cap_request_at"] = time.monotonic()

def fetch_historical_market_cap_series(
    symbol,
    cache_dir,
    limit=None,
    from_date=None,
    to_date=None,
    refresh=False,
    requery_incomplete=True,
    request_pause_seconds=0.0,
    session=None,
):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = _market_cap_cache_path(cache_dir, symbol)
    cached_series = pd.Series(dtype="float64", name=symbol)

    if cache_path.exists() and not refresh:
        cached_frame = pd.read_csv(cache_path)
        cached_series = _market_cap_series_from_frame(cached_frame, symbol)
        if not cached_series.empty:
            requested_start = pd.Timestamp(from_date) if from_date else None
            requested_end = pd.Timestamp(to_date) if to_date else None
            covers_start = requested_start is None or cached_series.index.min() <= requested_start
            covers_end = requested_end is None or cached_series.index.max() >= requested_end - pd.Timedelta(days=7)
            if covers_start and covers_end:
                cached_series.attrs["market_cap_cache_status"] = "cache"
                return cached_series
            if not requery_incomplete or _market_cap_request_was_attempted(cache_dir, symbol, limit, from_date, to_date):
                cached_series.attrs["market_cap_cache_status"] = "partial_cache"
                return cached_series
    elif not refresh and _market_cap_request_was_attempted(cache_dir, symbol, limit, from_date, to_date):
        cached_series.attrs["market_cap_cache_status"] = "known_empty_cache"
        return cached_series

    _respect_market_cap_request_pause(request_pause_seconds)
    rows = fetch_fmp_historical_market_capitalization(
        symbol,
        limit=limit,
        from_date=from_date,
        to_date=to_date,
        session=session,
    )
    fetched_series = _market_cap_series_from_frame(pd.DataFrame(rows), symbol)
    _append_market_cap_request_log(cache_dir, symbol, limit, from_date, to_date, fetched_series)

    if not cached_series.empty and not fetched_series.empty:
        market_cap_series = pd.concat([cached_series, fetched_series]).sort_index()
        market_cap_series = market_cap_series[~market_cap_series.index.duplicated(keep="last")]
    elif not fetched_series.empty:
        market_cap_series = fetched_series
    else:
        market_cap_series = cached_series

    if not market_cap_series.empty:
        market_cap_series.to_frame("Market Cap").to_csv(cache_path, index_label="Date")
    market_cap_series.attrs["market_cap_cache_status"] = "fetched" if not fetched_series.empty else "empty_fetch"
    return market_cap_series

def fetch_historical_market_cap_frame(
    symbols,
    cache_dir,
    limit=None,
    from_date=None,
    to_date=None,
    refresh=False,
    requery_incomplete=True,
    request_pause_seconds=0.0,
    session=None,
):
    market_cap_series = []
    failures = {}
    cache_stats = {
        "cache": 0,
        "partial_cache": 0,
        "known_empty_cache": 0,
        "fetched": 0,
        "empty_fetch": 0,
    }
    for symbol in [symbol for symbol in dict.fromkeys(symbols) if symbol]:
        try:
            series = fetch_historical_market_cap_series(
                symbol,
                cache_dir=cache_dir,
                limit=limit,
                from_date=from_date,
                to_date=to_date,
                refresh=refresh,
                requery_incomplete=requery_incomplete,
                request_pause_seconds=request_pause_seconds,
                session=session,
            )
        except ValueError:
            raise
        except Exception as exc:
            failures[symbol] = str(exc)
            continue

        cache_status = series.attrs.get("market_cap_cache_status", "cache")
        cache_stats[cache_status] = cache_stats.get(cache_status, 0) + 1
        if series.empty:
            failures[symbol] = "No historical market-cap rows returned."
        else:
            market_cap_series.append(series)

    if not market_cap_series:
        raise ValueError("FMP did not return historical market caps for any sector peers.")
    market_cap_frame = pd.concat(market_cap_series, axis=1).sort_index()
    return market_cap_frame, failures, cache_stats

_market_cap_index_input_cache = {}

def build_market_cap_weighted_index(price_data, market_cap_data, symbols, index_name, base_value=100.0):
    available = [
        symbol
        for symbol in dict.fromkeys(symbols)
        if symbol in price_data.columns and symbol in market_cap_data.columns
    ]
    if len(available) < 2:
        raise ValueError(f"Market-cap weighted index needs at least 2 symbols with prices and market caps; found {len(available)}.")

    cache_key = (id(price_data), id(market_cap_data))
    cached_inputs = _market_cap_index_input_cache.get(cache_key)
    if cached_inputs is None:
        cached_prices = price_data.copy().where(price_data > 0).ffill()
        cached_inputs = {
            "prices": cached_prices,
            "log_returns": np.log(cached_prices).diff().replace([np.inf, -np.inf], np.nan),
            "market_caps": market_cap_data.copy().where(market_cap_data > 0).reindex(price_data.index).ffill(),
        }
        _market_cap_index_input_cache.clear()
        _market_cap_index_input_cache[cache_key] = cached_inputs

    candidate_prices = cached_inputs["prices"][available]
    candidate_market_caps = cached_inputs["market_caps"][available]
    price_observations = candidate_prices.notna().sum()
    market_cap_observations = candidate_market_caps.notna().sum()
    usable_symbols = [
        symbol
        for symbol in available
        if price_observations[symbol] >= 2 and market_cap_observations[symbol] >= 1
    ]
    if len(usable_symbols) < 2:
        raise ValueError(f"Market-cap weighted index needs at least 2 usable symbols; found {len(usable_symbols)}.")

    candidate_prices = candidate_prices[usable_symbols]
    candidate_market_caps = candidate_market_caps[usable_symbols]
    log_returns = cached_inputs["log_returns"][usable_symbols]
    aligned_market_caps = candidate_market_caps[usable_symbols]

    raw_weights = aligned_market_caps.div(aligned_market_caps.sum(axis=1), axis=0).shift(1)
    daily_weights = raw_weights.reindex(log_returns.index).where(log_returns.notna())
    daily_weight_sums = daily_weights.sum(axis=1, min_count=1)
    daily_weights = daily_weights.loc[daily_weight_sums.gt(0)]
    if daily_weights.empty:
        raise ValueError("Market-cap weights and price returns do not overlap enough to build an index.")

    daily_weights = daily_weights.div(daily_weights.sum(axis=1), axis=0)
    aligned_returns = log_returns.reindex(daily_weights.index).fillna(0.0)
    index_log_returns = aligned_returns.mul(daily_weights.fillna(0.0), axis=1).sum(axis=1)
    index_values = (base_value * np.exp(index_log_returns.cumsum())).rename(index_name)

    first_return_date = index_log_returns.index[0]
    first_return_position = candidate_prices.index.get_indexer([first_return_date])[0]
    base_date = candidate_prices.index[max(0, first_return_position - 1)]
    market_cap_weighted_index = pd.concat(
        [pd.Series([base_value], index=[base_date], name=index_name), index_values]
    ).sort_index()
    market_cap_weighted_index = market_cap_weighted_index[~market_cap_weighted_index.index.duplicated(keep="last")]

    latest_weights = daily_weights.iloc[-1].rename("Latest Market-Cap Weight")
    average_weights = daily_weights.mean().rename("Average Market-Cap Weight")
    latest_market_caps = aligned_market_caps.reindex(daily_weights.index).ffill().iloc[-1].rename("Latest Market Cap")
    weight_table = pd.concat([latest_market_caps, latest_weights, average_weights], axis=1)
    weight_table = weight_table.dropna(subset=["Latest Market-Cap Weight"]).sort_values(
        "Latest Market-Cap Weight",
        ascending=False,
    )
    return (
        market_cap_weighted_index,
        weight_table,
        index_log_returns.rename(f"{index_name} Log Return"),
        daily_weights,
    )

if price_frame.empty:
    raise ValueError("price_frame is empty. Run the price history fetch cell first.")

show_non_market_cap_indexes = globals().get("show_non_market_cap_indexes", False)

pca_end_date = price_frame.index.max()
pca_start_date = price_frame.index.min() if pca_years is None else pca_end_date - pd.DateOffset(years=pca_years)
sector_pca_price_frame = price_frame.loc[
    (price_frame.index >= pca_start_date) & (price_frame.index <= pca_end_date)
]

plot_end_date = pca_end_date
plot_start_date = sector_pca_price_frame.index.min() if plot_years is None else plot_end_date - pd.DateOffset(years=plot_years)
plot_window_label = "All Available History" if plot_years is None else f"{plot_years}Y"

sector_pca_symbols = sector_peer_universe["Normalized Symbol"].dropna().tolist()
sector_market_cap_symbols = [target_symbol, *sector_pca_symbols]
non_market_cap_index_variable_names = [
    "sector_pca_index",
    "sector_pca_loadings",
    "sector_pca_explained_variance_ratio",
    "sector_pca_log_returns",
    "sector_equal_weight_index",
    "sector_equal_weight_weights",
    "sector_equal_weight_log_returns",
    "sector_volatility_adjusted_equal_weight_index",
    "sector_volatility_adjusted_equal_weight_weights",
    "sector_volatility_adjusted_equal_weight_log_returns",
    "large_cap_sector_pca_index",
    "large_cap_sector_pca_loadings",
    "large_cap_sector_pca_explained_variance_ratio",
    "large_cap_sector_pca_log_returns",
    "dynamic_sector_pca_index",
    "dynamic_sector_pca_weights",
    "dynamic_sector_pca_summary",
    "dynamic_sector_pca_log_returns",
    "monthly_dynamic_sector_pca_index",
    "monthly_dynamic_sector_pca_weights",
    "monthly_dynamic_sector_pca_summary",
    "monthly_dynamic_sector_pca_log_returns",
    "large_cap_weekly_dynamic_sector_pca_index",
    "large_cap_weekly_dynamic_sector_pca_weights",
    "large_cap_weekly_dynamic_sector_pca_summary",
    "large_cap_weekly_dynamic_sector_pca_log_returns",
    "large_cap_monthly_dynamic_sector_pca_index",
    "large_cap_monthly_dynamic_sector_pca_weights",
    "large_cap_monthly_dynamic_sector_pca_summary",
    "large_cap_monthly_dynamic_sector_pca_log_returns",
]
if not show_non_market_cap_indexes:
    for variable_name in non_market_cap_index_variable_names:
        globals().pop(variable_name, None)

if show_non_market_cap_indexes:
    sector_pca_index, sector_pca_loadings, sector_pca_explained_variance_ratio, sector_pca_log_returns = build_pca_index(
        sector_pca_price_frame,
        sector_pca_symbols,
        index_name=f"{target_gics_row['Sector']} PCA Sector Index",
        min_history_ratio=1.0,
    )
    sector_equal_weight_index, sector_equal_weight_weights, sector_equal_weight_log_returns = build_equal_weight_index(
        sector_pca_price_frame,
        sector_pca_symbols,
        index_name=f"{target_gics_row['Sector']} Equal Weight Sector Index",
        min_history_ratio=1.0,
    )
    sector_volatility_adjusted_equal_weight_index, sector_volatility_adjusted_equal_weight_weights, sector_volatility_adjusted_equal_weight_log_returns = build_volatility_adjusted_equal_weight_index(
        sector_pca_price_frame,
        sector_pca_symbols,
        index_name=f"{target_gics_row['Sector']} Volatility-Adjusted Equal Weight Sector Index",
        min_history_ratio=1.0,
    )
market_cap_from_date = sector_pca_price_frame.index.min().strftime("%Y-%m-%d")
market_cap_to_date = sector_pca_price_frame.index.max().strftime("%Y-%m-%d")
fmp_session = requests.Session()
sector_historical_market_caps, sector_historical_market_cap_failures, sector_historical_market_cap_cache_stats = fetch_historical_market_cap_frame(
    sector_market_cap_symbols,
    cache_dir=market_cap_cache_dir,
    limit=market_cap_history_limit,
    from_date=market_cap_from_date,
    to_date=market_cap_to_date,
    refresh=refresh_market_cap_cache,
    requery_incomplete=requery_incomplete_market_cap_cache,
    request_pause_seconds=market_cap_request_pause_seconds,
    session=fmp_session,
)
selected_gics_market_cap_peer_frames = {
    "Sector": sector_peer_universe,
    "Industry Group": industry_group_peer_universe,
    "Industry": industry_peer_universe,
    "Sub-Industry": sub_industry_peer_universe,
}
market_cap_index_frame_definitions = {
    "Total Cap": None,
    "Large Cap": "Large Cap",
    "Mid Cap": "Mid Cap",
    "Small Cap": "Small Cap",
}
market_cap_weighted_index_frames = {}
market_cap_weighted_index_weights = {}
market_cap_weighted_index_log_returns = {}
market_cap_weighted_index_daily_weights = {}
market_cap_weighted_index_failures = {}
market_cap_weighted_index_summary_rows = []

def build_market_cap_weighted_index_panel(cap_bucket_name, capitalization_filter=None):
    panel_series = {}
    for gics_level, peer_frame in selected_gics_market_cap_peer_frames.items():
        if capitalization_filter is None:
            level_symbols = peer_frame["Normalized Symbol"].dropna().tolist()
        else:
            level_symbols = (
                peer_frame.loc[
                    peer_frame["Capitalization"].astype(str).eq(capitalization_filter),
                    "Normalized Symbol",
                ]
                .dropna()
                .tolist()
            )

        index_key = (cap_bucket_name, gics_level)
        if len(level_symbols) < 2:
            market_cap_weighted_index_failures[index_key] = f"Only {len(level_symbols)} symbols available."
            continue

        try:
            index_series, weights, log_returns, daily_weights = build_market_cap_weighted_index(
                sector_pca_price_frame,
                sector_historical_market_caps,
                level_symbols,
                index_name=f"{target_gics_row[gics_level]} {cap_bucket_name} Market-Cap Weighted {gics_level} Index",
            )
        except ValueError as exc:
            market_cap_weighted_index_failures[index_key] = str(exc)
            continue

        panel_series[gics_level] = index_series.rename(gics_level)
        market_cap_weighted_index_weights[index_key] = weights
        market_cap_weighted_index_log_returns[index_key] = log_returns
        market_cap_weighted_index_daily_weights[index_key] = daily_weights
        market_cap_weighted_index_summary_rows.append(
            {
                "Cap Bucket": cap_bucket_name,
                "GICS Level": gics_level,
                "GICS Name": target_gics_row[gics_level],
                "Symbols": len(weights),
                "Start Date": index_series.index.min(),
                "End Date": index_series.index.max(),
                "Largest Weight": weights["Latest Market-Cap Weight"].max(),
            }
        )

    if not panel_series:
        return pd.DataFrame()
    return pd.concat(panel_series, axis=1).sort_index().dropna(how="all")

for cap_bucket_name, capitalization_filter in market_cap_index_frame_definitions.items():
    market_cap_weighted_index_frames[cap_bucket_name] = build_market_cap_weighted_index_panel(
        cap_bucket_name,
        capitalization_filter=capitalization_filter,
    )

total_cap_market_cap_weighted_indices = market_cap_weighted_index_frames["Total Cap"]
large_cap_market_cap_weighted_indices = market_cap_weighted_index_frames["Large Cap"]
mid_cap_market_cap_weighted_indices = market_cap_weighted_index_frames["Mid Cap"]
small_cap_market_cap_weighted_indices = market_cap_weighted_index_frames["Small Cap"]
market_cap_weighted_index_summary = pd.DataFrame(market_cap_weighted_index_summary_rows)

# Persist all total-cap GICS indexes for downstream notebooks.
# GICS peer frames exclude the target by default, so no benchmark contains the asset being evaluated.
factor_gics_index_levels = ["Sector", "Industry Group", "Industry", "Sub-Industry"]
factor_gics_index_suffixes = {
    "Sector": "sector",
    "Industry Group": "industry_group",
    "Industry": "industry",
    "Sub-Industry": "sub_industry",
}
factor_peer_index_level = "Sub-Industry"
safe_peer_index_symbol = target_symbol.replace(".", "_").replace("/", "_")
peer_index_cache_dir.mkdir(parents=True, exist_ok=True)
factor_gics_index_cache_paths = {}
factor_gics_index_labels = {}

for factor_gics_level in factor_gics_index_levels:
    factor_gics_index = total_cap_market_cap_weighted_indices.get(
        factor_gics_level,
        pd.Series(dtype="float64", name=factor_gics_level),
    ).dropna().rename("Close")
    if factor_gics_index.empty:
        print(f"No Total Cap {factor_gics_level} index was available to export.")
        continue

    peer_suffix = " (Peer)" if factor_gics_level == factor_peer_index_level else ""
    factor_gics_label = f"{target_symbol} {factor_gics_level} Index{peer_suffix}"
    factor_gics_cache_path = peer_index_cache_dir / (
        f"{safe_peer_index_symbol}_{factor_gics_index_suffixes[factor_gics_level]}_index.csv"
    )
    factor_gics_export = factor_gics_index.to_frame()
    factor_gics_export["Target Symbol"] = target_symbol
    factor_gics_export["Benchmark Label"] = factor_gics_label
    factor_gics_export["GICS Level"] = factor_gics_level
    factor_gics_export["GICS Name"] = target_gics_row[factor_gics_level]
    factor_gics_export["Cap Bucket"] = "Total Cap"
    factor_gics_export["Is Peer Index"] = factor_gics_level == factor_peer_index_level
    factor_gics_export.to_csv(factor_gics_cache_path, index_label="Date")
    factor_gics_index_cache_paths[factor_gics_level] = factor_gics_cache_path
    factor_gics_index_labels[factor_gics_level] = factor_gics_label
    print(
        f"Saved {factor_gics_label} ({factor_gics_level}: "
        f"{target_gics_row[factor_gics_level]}) to {factor_gics_cache_path}"
    )

# Keep the original peer-index artifact as a compatibility fallback for older consumers.
factor_peer_index_label = factor_gics_index_labels.get(
    factor_peer_index_level, f"{target_symbol} Peer Index"
)
factor_peer_index = total_cap_market_cap_weighted_indices.get(
    factor_peer_index_level, pd.Series(dtype="float64", name=factor_peer_index_label)
).dropna().rename("Close")
factor_peer_index_cache_path = peer_index_cache_dir / f"{safe_peer_index_symbol}_peer_index.csv"
if not factor_peer_index.empty:
    factor_peer_index_export = factor_peer_index.to_frame()
    factor_peer_index_export["Target Symbol"] = target_symbol
    factor_peer_index_export["Benchmark Label"] = factor_peer_index_label
    factor_peer_index_export["GICS Level"] = factor_peer_index_level
    factor_peer_index_export["GICS Name"] = target_gics_row[factor_peer_index_level]
    factor_peer_index_export["Cap Bucket"] = "Total Cap"
    factor_peer_index_export.to_csv(factor_peer_index_cache_path, index_label="Date")

sector_market_cap_weighted_index = total_cap_market_cap_weighted_indices.get(
    "Sector",
    pd.Series(dtype="float64", name="Sector"),
).rename("Market-Cap Weighted Sector Index")
sector_market_cap_weighted_weights = market_cap_weighted_index_weights.get(("Total Cap", "Sector"), pd.DataFrame())
sector_market_cap_weighted_log_returns = market_cap_weighted_index_log_returns.get(
    ("Total Cap", "Sector"),
    pd.Series(dtype="float64", name="Market-Cap Weighted Sector Index Log Return"),
)
sector_market_cap_weighted_daily_weights = market_cap_weighted_index_daily_weights.get(("Total Cap", "Sector"), pd.DataFrame())
sector_capitalization_market_cap_weighted_indexes = {
    f"{cap_bucket} Sector": frame["Sector"].rename(f"{cap_bucket} Sector")
    for cap_bucket, frame in market_cap_weighted_index_frames.items()
    if cap_bucket != "Total Cap" and "Sector" in frame.columns
}
large_cap_sector_pca_symbols = (
    sector_peer_universe.loc[
        sector_peer_universe["Capitalization"].astype(str).eq("Large Cap"),
        "Normalized Symbol",
    ]
    .dropna()
    .tolist()
)
if show_non_market_cap_indexes:
    large_cap_sector_pca_index, large_cap_sector_pca_loadings, large_cap_sector_pca_explained_variance_ratio, large_cap_sector_pca_log_returns = build_pca_index(
        sector_pca_price_frame,
        large_cap_sector_pca_symbols,
        index_name=f"{target_gics_row['Sector']} Large Cap PCA Sector Index",
        min_history_ratio=1.0,
    )

dynamic_pca_min_quarter_observations = 20
dynamic_pca_min_month_observations = 15
dynamic_pca_min_week_observations = 4
dynamic_pca_fit_window_days = 252
dynamic_pca_min_fit_observations = 200

def fit_pca_weights_from_returns(return_frame):
    return_frame = return_frame.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="all").dropna()
    if return_frame.shape[0] < 2 or return_frame.shape[1] < 2:
        return None, np.nan

    return_stds = return_frame.std(ddof=0).replace(0, np.nan)
    standardized_returns = ((return_frame - return_frame.mean()) / return_stds).dropna(axis=1).dropna()
    if standardized_returns.shape[0] < 2 or standardized_returns.shape[1] < 2:
        return None, np.nan

    _, singular_values, components = np.linalg.svd(standardized_returns.to_numpy(), full_matrices=False)
    loadings = pd.Series(components[0], index=standardized_returns.columns, name="PC1 Loading")
    if loadings.sum() < 0:
        loadings = -loadings

    positive_loadings = loadings.clip(lower=0)
    if positive_loadings.sum() == 0:
        weights = loadings.abs() / loadings.abs().sum()
    else:
        weights = positive_loadings / positive_loadings.sum()
    weights.name = "Dynamic PCA Weight"

    explained_variance = singular_values ** 2
    explained_variance_ratio = float(explained_variance[0] / explained_variance.sum())
    return weights, explained_variance_ratio

def build_periodic_dynamic_pca_index(
    price_data,
    symbols,
    index_name,
    rebalance_frequency,
    period_label,
    base_value=100.0,
    min_period_observations=20,
    fit_window_days=252,
    min_fit_observations=200,
):
    available = [symbol for symbol in dict.fromkeys(symbols) if symbol in price_data.columns]
    if len(available) < 2:
        raise ValueError(f"Dynamic PCA needs at least 2 available symbols; found {len(available)}.")

    candidate_prices = price_data[available].dropna(axis=1, how="all").copy()
    positive_price_columns = candidate_prices.apply(lambda series: series.dropna().gt(0).all())
    candidate_prices = candidate_prices.loc[:, positive_price_columns]
    log_returns = np.log(candidate_prices).diff().replace([np.inf, -np.inf], np.nan).dropna(how="all")

    period_index_returns = []
    period_weight_rows = []
    period_summary_rows = []

    for period, period_returns in log_returns.groupby(log_returns.index.to_period(rebalance_frequency)):
        period_start = period_returns.index.min()
        fit_window_returns = log_returns.loc[log_returns.index < period_start].tail(fit_window_days)
        fit_returns = fit_window_returns.dropna(axis=1, thresh=min_fit_observations).dropna()
        if fit_returns.shape[0] < min_fit_observations:
            continue

        weights, explained_variance_ratio = fit_pca_weights_from_returns(fit_returns)
        if weights is None:
            continue

        period_returns = period_returns.dropna(axis=1, thresh=min_period_observations)
        applied_symbols = [symbol for symbol in weights.index if symbol in period_returns.columns]
        if len(applied_symbols) < 2:
            continue
        applied_weights = weights.loc[applied_symbols]
        if applied_weights.sum() == 0:
            continue
        applied_weights = applied_weights / applied_weights.sum()
        applied_returns = period_returns[applied_symbols].fillna(0.0).mul(applied_weights, axis=1).sum(axis=1)
        period_index_returns.append(applied_returns)

        period_name = str(period)
        period_summary_rows.append(
            {
                period_label: period_name,
                "Start Date": period_returns.index.min(),
                "End Date": period_returns.index.max(),
                "Fit Start Date": fit_returns.index.min(),
                "Fit End Date": fit_returns.index.max(),
                "Fit Observations": len(fit_returns),
                "Symbols": len(applied_weights),
                "PC1 Explained Variance": explained_variance_ratio,
            }
        )
        period_weight_rows.extend(
            {
                period_label: period_name,
                "Symbol": symbol,
                "Dynamic PCA Weight": weight,
            }
            for symbol, weight in applied_weights.items()
        )

    if not period_index_returns:
        raise ValueError(f"No {period_label.lower()} PCA windows had enough data to build a dynamic index.")

    dynamic_log_returns = pd.concat(period_index_returns).sort_index()
    dynamic_values = (base_value * np.exp(dynamic_log_returns.cumsum())).rename(index_name)
    dynamic_index = pd.concat(
        [pd.Series([base_value], index=candidate_prices.index[:1], name=index_name), dynamic_values]
    ).sort_index()

    return (
        dynamic_index,
        pd.DataFrame(period_weight_rows),
        pd.DataFrame(period_summary_rows),
        dynamic_log_returns.rename(f"{index_name} Log Return"),
    )

if show_non_market_cap_indexes:
    dynamic_sector_pca_index, dynamic_sector_pca_weights, dynamic_sector_pca_summary, dynamic_sector_pca_log_returns = build_periodic_dynamic_pca_index(
        sector_pca_price_frame,
        sector_pca_symbols,
        index_name=f"{target_gics_row['Sector']} Quarterly Dynamic PCA Sector Index",
        rebalance_frequency="Q",
        period_label="Quarter",
        min_period_observations=dynamic_pca_min_quarter_observations,
        fit_window_days=dynamic_pca_fit_window_days,
        min_fit_observations=dynamic_pca_min_fit_observations,
    )
    monthly_dynamic_sector_pca_index, monthly_dynamic_sector_pca_weights, monthly_dynamic_sector_pca_summary, monthly_dynamic_sector_pca_log_returns = build_periodic_dynamic_pca_index(
        sector_pca_price_frame,
        sector_pca_symbols,
        index_name=f"{target_gics_row['Sector']} Monthly Dynamic PCA Sector Index",
        rebalance_frequency="M",
        period_label="Month",
        min_period_observations=dynamic_pca_min_month_observations,
        fit_window_days=dynamic_pca_fit_window_days,
        min_fit_observations=dynamic_pca_min_fit_observations,
    )
    large_cap_weekly_dynamic_sector_pca_index, large_cap_weekly_dynamic_sector_pca_weights, large_cap_weekly_dynamic_sector_pca_summary, large_cap_weekly_dynamic_sector_pca_log_returns = build_periodic_dynamic_pca_index(
        sector_pca_price_frame,
        large_cap_sector_pca_symbols,
        index_name=f"{target_gics_row['Sector']} Large Cap Weekly Dynamic PCA Sector Index",
        rebalance_frequency="W",
        period_label="Week",
        min_period_observations=dynamic_pca_min_week_observations,
        fit_window_days=dynamic_pca_fit_window_days,
        min_fit_observations=dynamic_pca_min_fit_observations,
    )
    large_cap_monthly_dynamic_sector_pca_index, large_cap_monthly_dynamic_sector_pca_weights, large_cap_monthly_dynamic_sector_pca_summary, large_cap_monthly_dynamic_sector_pca_log_returns = build_periodic_dynamic_pca_index(
        sector_pca_price_frame,
        large_cap_sector_pca_symbols,
        index_name=f"{target_gics_row['Sector']} Large Cap Monthly Dynamic PCA Sector Index",
        rebalance_frequency="M",
        period_label="Month",
        min_period_observations=dynamic_pca_min_month_observations,
        fit_window_days=dynamic_pca_fit_window_days,
        min_fit_observations=dynamic_pca_min_fit_observations,
    )

def normalize_price_series(series, name, base_value=100.0):
    series = series.dropna()
    if series.empty:
        return pd.Series(dtype="float64", name=name)
    return (series / series.iloc[0] * base_value).rename(name)

comparison_series = []
if target_symbol in sector_pca_price_frame.columns:
    comparison_series.append(
        normalize_price_series(sector_pca_price_frame[target_symbol], f"{target_symbol} Normalized Price")
    )
if sector_benchmark_symbol and sector_benchmark_symbol in sector_pca_price_frame.columns:
    comparison_series.append(
        normalize_price_series(
            sector_pca_price_frame[sector_benchmark_symbol], f"{sector_benchmark_symbol} Normalized Price"
        )
    )
market_cap_weighted_index_series = {}
for cap_bucket, index_frame in market_cap_weighted_index_frames.items():
    for gics_level in index_frame.columns:
        display_name = f"{cap_bucket} {gics_level}"
        market_cap_weighted_index_series[display_name] = index_frame[gics_level].rename(display_name)
comparison_series.extend(market_cap_weighted_index_series.values())
if show_non_market_cap_indexes:
    comparison_series.extend(
        [
            sector_pca_index,
            sector_equal_weight_index,
            sector_volatility_adjusted_equal_weight_index,
            large_cap_sector_pca_index,
            dynamic_sector_pca_index,
            monthly_dynamic_sector_pca_index,
            large_cap_weekly_dynamic_sector_pca_index,
            large_cap_monthly_dynamic_sector_pca_index,
        ]
    )
sector_pca_index_frame = pd.concat(comparison_series, axis=1).dropna(how="all")
if sector_pca_index_frame.empty:
    sector_pca_plot_frame = sector_pca_index_frame
else:
    sector_pca_plot_frame = sector_pca_index_frame.loc[
        (sector_pca_index_frame.index >= plot_start_date)
        & (sector_pca_index_frame.index <= plot_end_date)
    ]

print(
    "Market-cap weighted index panels built: "
    + ", ".join(
        f"{cap_bucket} ({len(index_frame.columns)} GICS levels)"
        for cap_bucket, index_frame in market_cap_weighted_index_frames.items()
        if not index_frame.empty
    )
)
if market_cap_weighted_index_failures:
    print(
        "Market-cap weighted indexes skipped: "
        + ", ".join(
            f"{cap_bucket} {gics_level}: {reason}"
            for (cap_bucket, gics_level), reason in market_cap_weighted_index_failures.items()
        )
    )
print(f"Analysis price window: {pca_start_date:%Y-%m-%d} to {pca_end_date:%Y-%m-%d}")
if not sector_historical_market_caps.empty:
    print(
        f"FMP market-cap window: {sector_historical_market_caps.index.min():%Y-%m-%d} to "
        f"{sector_historical_market_caps.index.max():%Y-%m-%d}"
    )
print(
    "FMP market-cap cache stats: "
    + ", ".join(f"{key}={value}" for key, value in sector_historical_market_cap_cache_stats.items() if value)
)
if not market_cap_weighted_index_summary.empty:
    print(
        f"Market-cap weighted index window: {market_cap_weighted_index_summary['Start Date'].min():%Y-%m-%d} to "
        f"{market_cap_weighted_index_summary['End Date'].max():%Y-%m-%d}"
    )
if sector_historical_market_cap_failures:
    print(f"FMP market-cap data unavailable for {len(sector_historical_market_cap_failures)} sector peers.")
if show_non_market_cap_indexes:
    print(f"Sector PCA index built from {len(sector_pca_loadings)} sector peers.")
    print(f"Equal-weight sector index built from {len(sector_equal_weight_weights)} sector peers.")
    print(f"Volatility-adjusted equal-weight sector index built from {len(sector_volatility_adjusted_equal_weight_weights)} sector peers.")
    print(f"Large-cap static PCA index built from {len(large_cap_sector_pca_loadings)} sector peers.")
    print(f"Dynamic PCA recomputed across {len(dynamic_sector_pca_summary)} quarters.")
    print(f"Monthly dynamic PCA recomputed across {len(monthly_dynamic_sector_pca_summary)} months.")
    print(f"Large-cap weekly dynamic PCA recomputed across {len(large_cap_weekly_dynamic_sector_pca_summary)} weeks.")
    print(f"Large-cap monthly dynamic PCA recomputed across {len(large_cap_monthly_dynamic_sector_pca_summary)} months.")
    print(f"Dynamic PCA fit window: trailing {dynamic_pca_fit_window_days} trading days.")
    print(f"PC1 explained variance: {sector_pca_explained_variance_ratio:.1%}")
    print(f"Large-cap PC1 explained variance: {large_cap_sector_pca_explained_variance_ratio:.1%}")
if not sector_pca_plot_frame.empty:
    print(
        f"Plot window: {sector_pca_plot_frame.index.min():%Y-%m-%d} to "
        f"{sector_pca_plot_frame.index.max():%Y-%m-%d}"
    )

sector_pca_fig = go.Figure()
for column in sector_pca_plot_frame.columns:
    sector_pca_fig.add_trace(
        go.Scatter(
            x=sector_pca_plot_frame.index,
            y=sector_pca_plot_frame[column],
            mode="lines",
            name=column,
            hovertemplate=f"{column}<br>%{{x|%Y-%m-%d}}<br>%{{y:.2f}}<extra></extra>",
        )
    )

benchmark_title_suffix = (
    f" and {sector_benchmark_symbol}"
    if sector_benchmark_symbol and sector_benchmark_symbol in sector_pca_price_frame.columns
    else ""
)
sector_pca_xaxis = dict(title="Date")
if not sector_pca_plot_frame.empty:
    sector_pca_xaxis["range"] = [
        plot_start_date,
        plot_end_date,
    ]
sector_pca_fig.update_layout(
    title=f"{target_symbol} vs Market-Cap Weighted GICS Index Panels{benchmark_title_suffix} ({plot_window_label})",
    template="plotly_dark",
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#f8fafc"),
    xaxis=sector_pca_xaxis,
    yaxis_title="Index Value (Base 100)",
    hovermode="x unified",
    margin=dict(t=64, r=24, b=48, l=64),
    height=520,
)
sector_pca_fig.show()



In [ ]:
# 5B. Plot latest GICS market-cap weighting
from plotly.subplots import make_subplots

if "market_cap_weighted_index_weights" not in globals():
    raise ValueError("Run block 5 first so latest market-cap weights are available.")

gics_market_cap_pie_levels = ["Sector", "Industry Group", "Industry", "Sub-Industry"]

def format_market_cap_label(value):
    if pd.isna(value):
        return "n/a"
    value = float(value)
    if abs(value) >= 1_000_000_000_000:
        return f"${value / 1_000_000_000_000:.2f}T"
    if abs(value) >= 1_000_000_000:
        return f"${value / 1_000_000_000:.1f}B"
    if abs(value) >= 1_000_000:
        return f"${value / 1_000_000:.1f}M"
    return f"${value:,.0f}"

def latest_market_cap_for_symbol(symbol):
    market_cap_series = pd.Series(dtype="float64", name=symbol)
    if "sector_historical_market_caps" in globals() and symbol in sector_historical_market_caps.columns:
        market_cap_series = sector_historical_market_caps[symbol].dropna()

    if market_cap_series.empty and "fetch_historical_market_cap_series" in globals():
        try:
            market_cap_series = fetch_historical_market_cap_series(
                symbol,
                cache_dir=market_cap_cache_dir,
                limit=market_cap_history_limit,
                from_date=market_cap_from_date,
                to_date=market_cap_to_date,
                refresh=refresh_market_cap_cache,
                requery_incomplete=requery_incomplete_market_cap_cache,
                request_pause_seconds=market_cap_request_pause_seconds,
            )
            if not market_cap_series.empty:
                globals()["sector_historical_market_caps"] = pd.concat(
                    [sector_historical_market_caps, market_cap_series.rename(symbol)],
                    axis=1,
                ).sort_index()
        except Exception as exc:
            print(f"Target market-cap history unavailable for {symbol}: {exc}")

    if market_cap_series.empty:
        return np.nan
    return pd.to_numeric(market_cap_series, errors="coerce").dropna().iloc[-1]

target_latest_market_cap = latest_market_cap_for_symbol(target_symbol)

def latest_market_cap_weight_table(gics_level):
    weight_table = market_cap_weighted_index_weights.get(("Total Cap", gics_level), pd.DataFrame()).copy()
    if weight_table.empty:
        weight_table = pd.DataFrame(columns=["Symbol", "Latest Market Cap"])
    else:
        weight_table = weight_table.reset_index()
        symbol_column = "Symbol" if "Symbol" in weight_table.columns else weight_table.columns[0]
        weight_table = weight_table.rename(columns={symbol_column: "Symbol"})
        weight_table = weight_table[["Symbol", "Latest Market Cap"]].copy()

    weight_table["Latest Market Cap"] = pd.to_numeric(weight_table["Latest Market Cap"], errors="coerce")
    weight_table = weight_table.dropna(subset=["Latest Market Cap"])
    weight_table = weight_table.loc[weight_table["Latest Market Cap"] > 0]
    if pd.notna(target_latest_market_cap) and target_latest_market_cap > 0:
        weight_table = weight_table.loc[~weight_table["Symbol"].astype(str).eq(target_symbol)].copy()
        weight_table = pd.concat(
            [
                pd.DataFrame(
                    [{"Symbol": target_symbol, "Latest Market Cap": target_latest_market_cap}]
                ),
                weight_table,
            ],
            ignore_index=True,
        )
    else:
        print(f"Target market cap unavailable for {target_symbol}; target slice cannot be added to {gics_level} pie.")

    total_market_cap = weight_table["Latest Market Cap"].sum()
    if total_market_cap <= 0:
        return pd.DataFrame(columns=["Symbol", "Latest Market Cap", "Latest Market-Cap Weight"])
    weight_table["Latest Market-Cap Weight"] = weight_table["Latest Market Cap"] / total_market_cap
    return weight_table.sort_values("Latest Market-Cap Weight", ascending=False)

gics_market_cap_pie_tables = {
    gics_level: latest_market_cap_weight_table(gics_level)
    for gics_level in gics_market_cap_pie_levels
}

gics_market_cap_weight_fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"type": "domain"}, {"type": "domain"}], [{"type": "domain"}, {"type": "domain"}]],
    subplot_titles=[
        f"{gics_level}: {target_gics_row[gics_level]}"
        for gics_level in gics_market_cap_pie_levels
    ],
)

for (gics_level, weights), (row, col) in zip(
    gics_market_cap_pie_tables.items(),
    [(1, 1), (1, 2), (2, 1), (2, 2)],
):
    if weights.empty:
        gics_market_cap_weight_fig.add_trace(
            go.Pie(
                labels=["No market-cap data"],
                values=[1],
                text=["No data"],
                textinfo="text",
                marker=dict(colors=["#475569"], line=dict(color="#0f172a", width=1)),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        continue

    symbols = weights["Symbol"].astype(str).tolist()
    market_caps = weights["Latest Market Cap"]
    weights_pct = weights["Latest Market-Cap Weight"] * 100
    visible_text = [
        f"{symbol}<br>{weight:.1f}%" if symbol == target_symbol or (pd.notna(weight) and weight >= 2) else ""
        for symbol, weight in zip(symbols, weights_pct)
    ]
    pull = [0.08 if symbol == target_symbol else 0 for symbol in symbols]
    customdata = pd.DataFrame(
        {
            "Weight": weights_pct,
            "Market Cap Label": market_caps.map(format_market_cap_label),
        }
    ).to_numpy()

    gics_market_cap_weight_fig.add_trace(
        go.Pie(
            labels=symbols,
            values=market_caps,
            text=visible_text,
            textinfo="text",
            customdata=customdata,
            hovertemplate=(
                "%{label}<br>"
                "Market cap: %{customdata[1]}<br>"
                "Weight: %{customdata[0]:.2f}%<extra></extra>"
            ),
            hole=0.32,
            sort=False,
            direction="clockwise",
            pull=pull,
            marker=dict(line=dict(color="#0f172a", width=1)),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

gics_market_cap_weight_fig.update_layout(
    title=f"{target_symbol} GICS Company Market-Cap Weights",
    template="plotly_dark",
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#f8fafc"),
    margin=dict(t=100, r=24, b=32, l=24),
    height=900,
    uniformtext_minsize=10,
    uniformtext_mode="hide",
)
gics_market_cap_weight_fig.show(config={"responsive": True, "displaylogo": False})

# Market-cap-sized treemap and company table reusing the block 5 market-cap frame only.
required_market_cap_treemap_objects = [
    "gics_peer_treemap_nodes",
    "gics_peer_treemap_source",
    "gics_peer_treemap_levels",
    "sector_historical_market_caps",
]
missing_market_cap_treemap_objects = [
    name for name in required_market_cap_treemap_objects if name not in globals()
]

if missing_market_cap_treemap_objects:
    print(
        "Market-cap treemap skipped; run blocks 3 and 5 first. Missing: "
        + ", ".join(missing_market_cap_treemap_objects)
    )
elif sector_historical_market_caps.empty:
    print("Market-cap treemap skipped: sector_historical_market_caps is empty.")
else:
    loaded_latest_market_cap_map = {}
    for symbol, market_cap_series in sector_historical_market_caps.items():
        latest_series = pd.to_numeric(market_cap_series, errors="coerce").dropna()
        if latest_series.empty:
            continue
        latest_value = float(latest_series.iloc[-1])
        if latest_value > 0:
            loaded_latest_market_cap_map[str(symbol)] = latest_value

    company_rows = gics_peer_treemap_source.drop_duplicates("Normalized Symbol").copy()
    missing_market_cap_symbols = sorted(
        set(company_rows["Normalized Symbol"].astype(str)) - set(loaded_latest_market_cap_map)
    )
    treemap_node_metadata = gics_peer_treemap_nodes.drop_duplicates("Node ID").set_index("Node ID")
    gics_market_cap_treemap_filter_specs = [
        ("All Caps", None),
        ("Large Cap", {"Large Cap"}),
        ("Mid Cap", {"Mid Cap"}),
        ("Small Cap", {"Small Cap"}),
        ("Large + Mid", {"Large Cap", "Mid Cap"}),
        ("Mid + Small", {"Mid Cap", "Small Cap"}),
    ]

    def summarize_filtered_capitalization(group):
        counts = group["Capitalization"].value_counts()
        ordered_buckets = ["Large Cap", "Mid Cap", "Small Cap", "Unclassified"]
        lines = [
            f"{bucket}: {int(counts[bucket])}"
            for bucket in ordered_buckets
            if bucket in counts.index
        ]
        return "Capitalization:<br>" + "<br>".join(lines) if lines else "Capitalization: Unclassified"

    def metadata_value(node_id, column, fallback=""):
        return treemap_node_metadata.at[node_id, column] if node_id in treemap_node_metadata.index else fallback

    def build_filtered_market_cap_treemap_nodes(filter_label, allowed_capitalizations):
        filtered_company_rows = company_rows.copy()
        if allowed_capitalizations is not None:
            filtered_company_rows = filtered_company_rows.loc[
                filtered_company_rows["Capitalization"].astype(str).isin(allowed_capitalizations)
            ].copy()

        filtered_company_rows["Latest Market Cap"] = (
            filtered_company_rows["Normalized Symbol"].astype(str).map(loaded_latest_market_cap_map)
        )
        filtered_company_rows["Latest Market Cap"] = pd.to_numeric(
            filtered_company_rows["Latest Market Cap"],
            errors="coerce",
        )
        filtered_company_rows = filtered_company_rows.loc[
            filtered_company_rows["Latest Market Cap"].gt(0)
        ].copy()

        filtered_treemap_rows = []
        for level_index, level_name in enumerate(gics_peer_treemap_levels):
            path_columns = gics_peer_treemap_levels[: level_index + 1]
            for path_values, group in filtered_company_rows.groupby(path_columns, sort=True):
                if not isinstance(path_values, tuple):
                    path_values = (path_values,)
                path_values = tuple(map(str, path_values))
                node_id = " > ".join(path_values)
                if node_id not in treemap_node_metadata.index:
                    continue
                capitalization_text = summarize_filtered_capitalization(group)
                if "Direct Peer Container: Yes" in str(metadata_value(node_id, "Capitalization Text")):
                    capitalization_text += "<br>Direct Peer Container: Yes"
                latest_market_cap = float(group["Latest Market Cap"].sum())
                filtered_treemap_rows.append(
                    {
                        "Node ID": node_id,
                        "Parent ID": metadata_value(node_id, "Parent ID"),
                        "Label": metadata_value(node_id, "Label", path_values[-1]),
                        "GICS Level": metadata_value(node_id, "GICS Level"),
                        "Path Text": metadata_value(node_id, "Path Text"),
                        "Capitalization Text": capitalization_text,
                        "Company Count": int(group["Normalized Symbol"].nunique()),
                        "Color": metadata_value(node_id, "Color", "#9ca3af"),
                        "Border Color": metadata_value(node_id, "Border Color", "#111827"),
                        "Border Width": metadata_value(node_id, "Border Width", 1),
                        "Text Color": metadata_value(node_id, "Text Color", "#f8fafc"),
                        "Latest Market Cap": latest_market_cap,
                        "Filter": filter_label,
                    }
                )

        for _, row in filtered_company_rows.iterrows():
            symbol = str(row["Normalized Symbol"])
            parent_values = tuple(str(row[column]) for column in gics_peer_treemap_levels)
            node_id = f"{' > '.join(parent_values)} > {symbol}"
            if node_id not in treemap_node_metadata.index:
                continue
            filtered_treemap_rows.append(
                {
                    "Node ID": node_id,
                    "Parent ID": metadata_value(node_id, "Parent ID"),
                    "Label": metadata_value(node_id, "Label", symbol),
                    "GICS Level": "Company",
                    "Path Text": metadata_value(node_id, "Path Text"),
                    "Capitalization Text": metadata_value(node_id, "Capitalization Text"),
                    "Company Count": 1,
                    "Color": metadata_value(node_id, "Color", "#9ca3af"),
                    "Border Color": metadata_value(node_id, "Border Color", "#111827"),
                    "Border Width": metadata_value(node_id, "Border Width", 1),
                    "Text Color": metadata_value(node_id, "Text Color", "#f8fafc"),
                    "Latest Market Cap": float(row["Latest Market Cap"]),
                    "Filter": filter_label,
                    "Symbol": symbol,
                    "Sector": str(row["Root"]),
                    "Industry Group": str(row["Industry Group"]),
                    "Industry": str(row["Industry"]),
                    "Sub-Industry": str(row["Sub-Industry"]),
                }
            )

        filtered_nodes = pd.DataFrame(filtered_treemap_rows)
        if filtered_nodes.empty:
            return filtered_nodes
        filtered_nodes = filtered_nodes.loc[
            pd.to_numeric(filtered_nodes["Latest Market Cap"], errors="coerce").gt(0)
        ].copy()
        filtered_nodes["Market Cap Label"] = filtered_nodes["Latest Market Cap"].map(format_market_cap_label)
        filtered_nodes["Display Text"] = (
            filtered_nodes["Label"].astype(str) + "<br>" + filtered_nodes["Market Cap Label"].astype(str)
        )
        return filtered_nodes

    def market_cap_treemap_trace(filtered_nodes, filter_label, visible=False):
        return go.Treemap(
            ids=filtered_nodes["Node ID"],
            labels=filtered_nodes["Label"],
            parents=filtered_nodes["Parent ID"],
            values=filtered_nodes["Latest Market Cap"],
            branchvalues="total",
            visible=visible,
            name=filter_label,
            marker=dict(
                colors=filtered_nodes["Color"],
                line=dict(
                    color=filtered_nodes["Border Color"],
                    width=filtered_nodes["Border Width"],
                ),
            ),
            text=filtered_nodes["Display Text"],
            customdata=filtered_nodes[
                ["GICS Level", "Path Text", "Capitalization Text", "Market Cap Label", "Company Count", "Filter"]
            ],
            textinfo="text",
            textfont=dict(color=filtered_nodes["Text Color"].tolist()),
            hovertemplate=(
                "<b>%{label}</b><br>"
                "Filter: %{customdata[5]}<br>"
                "Level: %{customdata[0]}<br>"
                "%{customdata[1]}<br>"
                "Market cap: %{customdata[3]}<br>"
                "Companies: %{customdata[4]}<br>"
                "%{customdata[2]}<extra></extra>"
            ),
            maxdepth=5,
            pathbar=dict(visible=True),
        )

    def market_cap_company_table(filtered_nodes):
        company_table = filtered_nodes.loc[filtered_nodes["GICS Level"].eq("Company")].copy()
        if company_table.empty:
            return company_table
        company_table["Latest Market Cap"] = pd.to_numeric(
            company_table["Latest Market Cap"],
            errors="coerce",
        )
        company_table = company_table.dropna(subset=["Latest Market Cap"])
        company_table = company_table.loc[company_table["Latest Market Cap"].gt(0)].copy()
        company_table = company_table.sort_values("Latest Market Cap", ascending=False)
        total_market_cap = company_table["Latest Market Cap"].sum()
        company_table["Market-Cap Weight"] = (
            company_table["Latest Market Cap"] / total_market_cap if total_market_cap > 0 else np.nan
        )
        company_table["Market-Cap Weight Label"] = company_table["Market-Cap Weight"].map(
            lambda value: f"{value:.1%}" if pd.notna(value) else "n/a"
        )
        return company_table

    def treemap_style_for_node_id(node_id):
        if node_id in treemap_node_metadata.index:
            return (
                str(treemap_node_metadata.at[node_id, "Color"]),
                str(treemap_node_metadata.at[node_id, "Text Color"]),
            )
        return "#0f172a", "#e2e8f0"

    def treemap_style_for_table_level(row, level_name):
        sector = str(row["Sector"])
        industry_group = str(row["Industry Group"])
        industry = str(row["Industry"])
        sub_industry = str(row["Sub-Industry"])
        node_ids = {
            "Sector": sector,
            "Industry Group": f"{sector} > {industry_group}",
            "Industry": f"{sector} > {industry_group} > {industry}",
            "Sub-Industry": f"{sector} > {industry_group} > {industry} > {sub_industry}",
        }
        return treemap_style_for_node_id(node_ids[level_name])

    def table_level_color_lists(company_table, level_name):
        styles = [
            treemap_style_for_table_level(row, level_name)
            for _, row in company_table.iterrows()
        ]
        return [style[0] for style in styles], [style[1] for style in styles]

    def market_cap_table_trace(filtered_nodes, filter_label, visible=False):
        company_table = market_cap_company_table(filtered_nodes)
        if company_table.empty:
            table_values = [["No companies with loaded market caps"], [""], [""], [""], [""], [""]]
            table_fill_color = [["#0f172a"] for _ in range(6)]
            table_font_color = [["#e2e8f0"] for _ in range(6)]
        else:
            table_values = [
                company_table["Symbol"].astype(str).tolist(),
                company_table["Market Cap Label"].astype(str).tolist(),
                company_table["Sector"].astype(str).tolist(),
                company_table["Industry"].astype(str).tolist(),
                company_table["Industry Group"].astype(str).tolist(),
                company_table["Sub-Industry"].astype(str).tolist(),
            ]
            row_fill_colors = company_table["Color"].astype(str).tolist()
            row_text_colors = company_table["Text Color"].astype(str).tolist()
            sector_fill_colors, sector_text_colors = table_level_color_lists(company_table, "Sector")
            industry_fill_colors, industry_text_colors = table_level_color_lists(company_table, "Industry")
            industry_group_fill_colors, industry_group_text_colors = table_level_color_lists(
                company_table,
                "Industry Group",
            )
            sub_industry_fill_colors, sub_industry_text_colors = table_level_color_lists(company_table, "Sub-Industry")
            table_fill_color = [
                row_fill_colors,
                row_fill_colors,
                sector_fill_colors,
                industry_fill_colors,
                industry_group_fill_colors,
                sub_industry_fill_colors,
            ]
            table_font_color = [
                row_text_colors,
                row_text_colors,
                sector_text_colors,
                industry_text_colors,
                industry_group_text_colors,
                sub_industry_text_colors,
            ]
        return go.Table(
            visible=visible,
            name=f"{filter_label} Table",
            columnwidth=[0.7, 0.9, 1.2, 1.8, 1.8, 1.9],
            header=dict(
                values=["Symbol", "Market Cap", "Sector", "Industry", "Industry Group", "Sub-Industry"],
                fill_color="#111827",
                line_color="#334155",
                font=dict(color="#f8fafc", size=12),
                align="left",
                height=28,
            ),
            cells=dict(
                values=table_values,
                fill_color=table_fill_color,
                line_color="#1e293b",
                font=dict(color=table_font_color, size=11),
                align="left",
                height=24,
            ),
        )

    gics_peer_market_cap_treemap_tables = {
        filter_label: build_filtered_market_cap_treemap_nodes(filter_label, allowed_capitalizations)
        for filter_label, allowed_capitalizations in gics_market_cap_treemap_filter_specs
    }
    gics_peer_market_cap_treemap_nodes = gics_peer_market_cap_treemap_tables.get(
        "All Caps",
        pd.DataFrame(),
    )
    gics_peer_market_cap_company_tables = {
        filter_label: market_cap_company_table(filtered_nodes)
        for filter_label, filtered_nodes in gics_peer_market_cap_treemap_tables.items()
    }
    nonempty_treemap_filters = [
        filter_label
        for filter_label, filtered_nodes in gics_peer_market_cap_treemap_tables.items()
        if not filtered_nodes.empty
    ]
    empty_treemap_filters = [
        filter_label
        for filter_label, filtered_nodes in gics_peer_market_cap_treemap_tables.items()
        if filtered_nodes.empty
    ]

    if not nonempty_treemap_filters:
        print("Dash treemap inputs skipped: no loaded market caps matched the GICS treemap symbols.")
    else:
        print(
            "Prepared Dash treemap/table inputs from the already-loaded sector_historical_market_caps frame; "
            "no extra FMP calls are made here."
        )
        if missing_market_cap_symbols:
            print(
                f"Dash treemap/table omitted {len(missing_market_cap_symbols)} symbols without loaded caps: "
                + ", ".join(missing_market_cap_symbols[:12])
                + ("..." if len(missing_market_cap_symbols) > 12 else "")
            )
        if empty_treemap_filters:
            print("Dash treemap/table filters with no loaded data: " + ", ".join(empty_treemap_filters))

gics_market_cap_pie_tables["Sector"].head(15)


In [ ]:
# 5D. Dash click-to-filter market-cap treemap
import importlib

from Quantapp.visualization.views.single_asset_profile.pricing import peer_market_cap_dash

peer_market_cap_dash = importlib.reload(peer_market_cap_dash)
create_peer_market_cap_dash_app = peer_market_cap_dash.create_peer_market_cap_dash_app

required_dash_treemap_objects = [
    "gics_peer_market_cap_treemap_tables",
    "gics_peer_market_cap_company_tables",
    "nonempty_treemap_filters",
    "market_cap_treemap_trace",
    "gics_peer_treemap_nodes",
    "target_symbol",
]
missing_dash_treemap_objects = [name for name in required_dash_treemap_objects if name not in globals()]
if missing_dash_treemap_objects:
    raise ValueError(
        "Run block 5B first so the Dash treemap inputs are available: "
        + ", ".join(missing_dash_treemap_objects)
    )

dash_treemap_app = create_peer_market_cap_dash_app(
    target_symbol=target_symbol,
    treemap_tables=gics_peer_market_cap_treemap_tables,
    company_tables=gics_peer_market_cap_company_tables,
    filter_labels=nonempty_treemap_filters,
    treemap_trace_builder=market_cap_treemap_trace,
    treemap_nodes=gics_peer_treemap_nodes,
)

dash_treemap_port = 8050
dash_treemap_run_kwargs = dict(port=dash_treemap_port, debug=False, use_reloader=False)
if hasattr(dash_treemap_app, "run"):
    try:
        dash_treemap_app.run(jupyter_mode="inline", jupyter_height=1500, **dash_treemap_run_kwargs)
    except TypeError:
        dash_treemap_app.run_server(**dash_treemap_run_kwargs)
else:
    dash_treemap_app.run_server(**dash_treemap_run_kwargs)


In [ ]:
# 5C. Plot nested GICS market-cap weighting down the target path
from plotly.subplots import make_subplots
import plotly.graph_objects as go

required_nested_gics_objects = [
    "sector_peer_universe",
    "target_gics_row",
    "target_symbol",
    "market_cap_weighted_index_weights",
]
missing_nested_gics_objects = [name for name in required_nested_gics_objects if name not in globals()]
if missing_nested_gics_objects:
    raise ValueError(
        "Run blocks 3 and 5 first so GICS peers and latest market-cap weights are available: "
        + ", ".join(missing_nested_gics_objects)
    )

if "format_market_cap_label" not in globals():
    def format_market_cap_label(value):
        if pd.isna(value):
            return "n/a"
        value = float(value)
        if abs(value) >= 1_000_000_000_000:
            return f"${value / 1_000_000_000_000:.2f}T"
        if abs(value) >= 1_000_000_000:
            return f"${value / 1_000_000_000:.1f}B"
        if abs(value) >= 1_000_000:
            return f"${value / 1_000_000:.1f}M"
        return f"${value:,.0f}"


def normalized_weight_table_symbols(weight_table):
    if weight_table.empty:
        return pd.DataFrame(columns=["Symbol", "Latest Market Cap"])

    normalized = weight_table.reset_index().copy()
    symbol_column = "Symbol" if "Symbol" in normalized.columns else normalized.columns[0]
    normalized = normalized.rename(columns={symbol_column: "Symbol"})
    normalized = normalized[["Symbol", "Latest Market Cap"]].copy()
    normalized["Symbol"] = normalized["Symbol"].astype(str)
    normalized["Latest Market Cap"] = pd.to_numeric(normalized["Latest Market Cap"], errors="coerce")
    normalized = normalized.dropna(subset=["Latest Market Cap"])
    normalized = normalized.loc[normalized["Latest Market Cap"] > 0]
    return normalized


def latest_market_cap_for_nested_symbol(symbol):
    if "latest_market_cap_for_symbol" in globals():
        return latest_market_cap_for_symbol(symbol)

    market_cap_series = pd.Series(dtype="float64", name=symbol)
    if "sector_historical_market_caps" in globals() and symbol in sector_historical_market_caps.columns:
        market_cap_series = sector_historical_market_caps[symbol].dropna()

    if market_cap_series.empty and "fetch_historical_market_cap_series" in globals():
        try:
            market_cap_series = fetch_historical_market_cap_series(
                symbol,
                cache_dir=market_cap_cache_dir,
                limit=market_cap_history_limit,
                from_date=market_cap_from_date,
                to_date=market_cap_to_date,
                refresh=refresh_market_cap_cache,
                requery_incomplete=requery_incomplete_market_cap_cache,
                request_pause_seconds=market_cap_request_pause_seconds,
            )
            if not market_cap_series.empty and "sector_historical_market_caps" in globals():
                globals()["sector_historical_market_caps"] = pd.concat(
                    [sector_historical_market_caps, market_cap_series.rename(symbol)],
                    axis=1,
                ).sort_index()
        except Exception as exc:
            print(f"Market-cap history unavailable for {symbol}: {exc}")

    if market_cap_series.empty:
        return np.nan
    return pd.to_numeric(market_cap_series, errors="coerce").dropna().iloc[-1]


sector_weight_table = normalized_weight_table_symbols(
    market_cap_weighted_index_weights.get(("Total Cap", "Sector"), pd.DataFrame())
)
nested_latest_market_cap_map = dict(
    zip(sector_weight_table["Symbol"], sector_weight_table["Latest Market Cap"])
)

target_nested_market_cap = globals().get("target_latest_market_cap", np.nan)
if pd.isna(target_nested_market_cap) or target_nested_market_cap <= 0:
    target_nested_market_cap = latest_market_cap_for_nested_symbol(target_symbol)
if pd.notna(target_nested_market_cap) and target_nested_market_cap > 0:
    nested_latest_market_cap_map[target_symbol] = float(target_nested_market_cap)

nested_target_row = target_gics_row.to_frame().T.copy()
nested_target_row["Normalized Symbol"] = target_symbol
nested_target_row["YFinance Symbol"] = target_symbol

nested_gics_market_cap_source = pd.concat(
    [nested_target_row, sector_peer_universe],
    ignore_index=True,
).drop_duplicates("Normalized Symbol")
nested_gics_market_cap_source["Normalized Symbol"] = (
    nested_gics_market_cap_source["Normalized Symbol"].astype(str)
)
nested_gics_market_cap_source["Latest Market Cap"] = (
    nested_gics_market_cap_source["Normalized Symbol"].map(nested_latest_market_cap_map)
)

missing_market_cap_symbols = nested_gics_market_cap_source.loc[
    nested_gics_market_cap_source["Latest Market Cap"].isna(),
    "Normalized Symbol",
].tolist()
for symbol in missing_market_cap_symbols:
    latest_market_cap = latest_market_cap_for_nested_symbol(symbol)
    if pd.notna(latest_market_cap) and latest_market_cap > 0:
        nested_latest_market_cap_map[symbol] = float(latest_market_cap)

nested_gics_market_cap_source["Latest Market Cap"] = (
    nested_gics_market_cap_source["Normalized Symbol"].map(nested_latest_market_cap_map)
)

nested_gics_market_cap_source["Latest Market Cap"] = pd.to_numeric(
    nested_gics_market_cap_source["Latest Market Cap"],
    errors="coerce",
)
nested_gics_market_cap_source = nested_gics_market_cap_source.dropna(subset=["Latest Market Cap"])
nested_gics_market_cap_source = nested_gics_market_cap_source.loc[
    nested_gics_market_cap_source["Latest Market Cap"] > 0
].copy()

nested_gics_market_cap_configs = [
    {
        "parent_level": "Sector",
        "parent_value": target_gics_row["Sector"],
        "child_level": "Industry Group",
        "target_child": target_gics_row["Industry Group"],
        "title": f"Sector: {target_gics_row['Sector']}<br>Industry Group Weights",
    },
    {
        "parent_level": "Industry Group",
        "parent_value": target_gics_row["Industry Group"],
        "child_level": "Industry",
        "target_child": target_gics_row["Industry"],
        "title": f"Industry Group: {target_gics_row['Industry Group']}<br>Industry Weights",
    },
    {
        "parent_level": "Industry",
        "parent_value": target_gics_row["Industry"],
        "child_level": "Sub-Industry",
        "target_child": target_gics_row["Sub-Industry"],
        "title": f"Industry: {target_gics_row['Industry']}<br>Sub-Industry Weights",
    },
    {
        "parent_level": "Sub-Industry",
        "parent_value": target_gics_row["Sub-Industry"],
        "child_level": "Company",
        "target_child": target_symbol,
        "title": f"Sub-Industry: {target_gics_row['Sub-Industry']}<br>Company Weights",
    },
]


def nested_gics_market_cap_weight_table(config):
    scoped = nested_gics_market_cap_source.loc[
        nested_gics_market_cap_source[config["parent_level"]].astype(str).eq(str(config["parent_value"]))
    ].copy()

    if scoped.empty:
        return pd.DataFrame(
            columns=[
                "Nested Slice",
                "Latest Market Cap",
                "Company Count",
                "Latest Market-Cap Weight",
            ]
        )

    if config["child_level"] == "Company":
        scoped["Nested Slice"] = scoped["Normalized Symbol"].astype(str)
    else:
        scoped["Nested Slice"] = scoped[config["child_level"]].fillna("Unclassified").astype(str)

    nested_table = (
        scoped.groupby("Nested Slice", dropna=False)
        .agg(
            **{
                "Latest Market Cap": ("Latest Market Cap", "sum"),
                "Company Count": ("Normalized Symbol", "nunique"),
            }
        )
        .reset_index()
    )
    nested_table = nested_table.loc[nested_table["Latest Market Cap"] > 0].copy()
    total_market_cap = nested_table["Latest Market Cap"].sum()

    if total_market_cap <= 0:
        return pd.DataFrame(
            columns=[
                "Nested Slice",
                "Latest Market Cap",
                "Company Count",
                "Latest Market-Cap Weight",
            ]
        )

    nested_table["Latest Market-Cap Weight"] = nested_table["Latest Market Cap"] / total_market_cap
    return nested_table.sort_values("Latest Market-Cap Weight", ascending=False)


nested_gics_market_cap_tables = {
    config["child_level"]: nested_gics_market_cap_weight_table(config)
    for config in nested_gics_market_cap_configs
}

nested_gics_market_cap_fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"type": "domain"}, {"type": "domain"}], [{"type": "domain"}, {"type": "domain"}]],
    subplot_titles=[config["title"] for config in nested_gics_market_cap_configs],
)

nested_gics_color_palette = [
    "#38BDF8",
    "#A78BFA",
    "#34D399",
    "#F59E0B",
    "#F472B6",
    "#22D3EE",
    "#F87171",
    "#C084FC",
    "#60A5FA",
    "#FBBF24",
    "#2DD4BF",
]

for config, (row, col) in zip(
    nested_gics_market_cap_configs,
    [(1, 1), (1, 2), (2, 1), (2, 2)],
):
    nested_table = nested_gics_market_cap_tables[config["child_level"]]

    if nested_table.empty:
        nested_gics_market_cap_fig.add_trace(
            go.Pie(
                labels=["No market-cap data"],
                values=[1],
                text=["No data"],
                textinfo="text",
                marker=dict(colors=["#475569"], line=dict(color="#0f172a", width=1)),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        continue

    labels = nested_table["Nested Slice"].astype(str).tolist()
    market_caps = nested_table["Latest Market Cap"]
    weights_pct = nested_table["Latest Market-Cap Weight"] * 100
    target_child = str(config["target_child"])
    colors = [
        "#38BDF8" if label == target_child else nested_gics_color_palette[idx % len(nested_gics_color_palette)]
        for idx, label in enumerate(labels)
    ]
    visible_text = [
        f"{label}<br>{weight:.1f}%" if label == target_child or weight >= 4 else ""
        for label, weight in zip(labels, weights_pct)
    ]
    pull = [0.08 if label == target_child else 0 for label in labels]
    customdata = pd.DataFrame(
        {
            "Weight": weights_pct,
            "Market Cap Label": market_caps.map(format_market_cap_label),
            "Company Count": nested_table["Company Count"],
            "Parent Level": config["parent_level"],
            "Parent Value": config["parent_value"],
            "Child Level": config["child_level"],
        }
    ).to_numpy()

    nested_gics_market_cap_fig.add_trace(
        go.Pie(
            labels=labels,
            values=market_caps,
            text=visible_text,
            textinfo="text",
            customdata=customdata,
            hovertemplate=(
                "%{label}<br>"
                "%{customdata[5]} market cap: %{customdata[1]}<br>"
                "Weight inside %{customdata[3]}: %{customdata[0]:.2f}%<br>"
                "Companies: %{customdata[2]:,.0f}<br>"
                "%{customdata[3]}: %{customdata[4]}<extra></extra>"
            ),
            hole=0.34,
            sort=False,
            direction="clockwise",
            pull=pull,
            marker=dict(colors=colors, line=dict(color="#0f172a", width=1)),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

nested_gics_market_cap_fig.update_layout(
    title=f"{target_symbol} Nested GICS Market-Cap Weights",
    template="plotly_dark",
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#f8fafc"),
    margin=dict(t=120, r=24, b=32, l=24),
    height=940,
    uniformtext_minsize=10,
    uniformtext_mode="hide",
)
nested_gics_market_cap_fig.show(config={"responsive": True, "displaylogo": False})

nested_gics_market_cap_summary = pd.concat(
    [
        table.assign(**{"Child Level": child_level})
        for child_level, table in nested_gics_market_cap_tables.items()
        if not table.empty
    ],
    ignore_index=True,
)
if not nested_gics_market_cap_summary.empty:
    nested_gics_market_cap_summary = nested_gics_market_cap_summary[
        ["Child Level", "Nested Slice", "Latest Market Cap", "Latest Market-Cap Weight", "Company Count"]
    ]

nested_gics_market_cap_summary.head(40)


In [ ]:
# 6. Plot rolling returns by GICS level for market-cap weighted indexes
return_horizon_days = 200
market_cap_gics_levels = ["Sector", "Industry Group", "Industry", "Sub-Industry"]
market_cap_gics_level_display_names = {
    gics_level: f"{gics_level}: {target_gics_row[gics_level]}"
    for gics_level in market_cap_gics_levels
}

if "market_cap_weighted_index_frames" not in globals():
    raise ValueError("market_cap_weighted_index_frames is not available. Run block 5 first.")

benchmark_series = None
if sector_benchmark_symbol and sector_benchmark_symbol in sector_pca_price_frame.columns:
    benchmark_series = normalize_price_series(
        sector_pca_price_frame[sector_benchmark_symbol], sector_benchmark_symbol
    )

sector_pca_200_day_returns_by_gics_level = {}
latest_sector_pca_200_day_return_rows = []
rolling_return_plot_years = 10
rolling_return_plot_end_date = plot_end_date
rolling_return_plot_start_date = max(
    plot_start_date,
    rolling_return_plot_end_date - pd.DateOffset(years=rolling_return_plot_years),
)

for gics_level in market_cap_gics_levels:
    rolling_return_series = {}
    if benchmark_series is not None:
        rolling_return_series[sector_benchmark_symbol] = benchmark_series

    for cap_bucket, index_frame in market_cap_weighted_index_frames.items():
        if gics_level not in index_frame.columns:
            continue
        gics_level_display_name = market_cap_gics_level_display_names[gics_level]
        display_name = f"{cap_bucket} {gics_level_display_name}"
        rolling_return_series[display_name] = index_frame[gics_level].rename(display_name)

    if len(rolling_return_series) <= (1 if benchmark_series is not None else 0):
        continue

    rolling_return_source = pd.concat(rolling_return_series, axis=1).sort_index()
    rolling_return_frame = rolling_return_source.pct_change(
        return_horizon_days,
        fill_method=None,
    )
    rolling_return_frame = rolling_return_frame.loc[
        (rolling_return_frame.index >= rolling_return_plot_start_date)
        & (rolling_return_frame.index <= rolling_return_plot_end_date)
    ].dropna(how="all")
    if rolling_return_frame.empty:
        continue

    sector_pca_200_day_returns_by_gics_level[gics_level] = rolling_return_frame
    latest_date = rolling_return_frame.index[-1]
    for index_name, latest_return in rolling_return_frame.tail(1).T[latest_date].dropna().items():
        latest_sector_pca_200_day_return_rows.append(
            {
                "GICS Level": market_cap_gics_level_display_names[gics_level],
                "Index": index_name,
                "Date": latest_date,
                f"{return_horizon_days}D Return": latest_return,
            }
        )

    rolling_return_fig = go.Figure()
    for column in rolling_return_frame.columns:
        return_series = rolling_return_frame[column].dropna()
        if return_series.empty:
            continue
        rolling_return_fig.add_trace(
            go.Scatter(
                x=return_series.index,
                y=return_series,
                mode="lines",
                name=column,
                hovertemplate=f"{column}<br>%{{x|%Y-%m-%d}}<br>%{{y:.2%}}<extra></extra>",
            )
        )

    rolling_return_fig.update_layout(
        title=f"{return_horizon_days}-Day Rolling Returns: {sector_benchmark_symbol} vs {market_cap_gics_level_display_names[gics_level]} Market-Cap Weighted Indexes",
        template="plotly_dark",
        paper_bgcolor="#0b0f14",
        plot_bgcolor="#0b0f14",
        font=dict(color="#f8fafc"),
        xaxis=dict(title="Date", range=[rolling_return_plot_start_date, rolling_return_plot_end_date]),
        yaxis=dict(title="Rolling Return", tickformat=".0%", zeroline=True, zerolinecolor="#94a3b8"),
        hovermode="x unified",
        margin=dict(t=64, r=24, b=48, l=64),
        height=520,
    )
    rolling_return_fig.show()

direct_peer_capitalization_buckets = ["Large Cap", "Mid Cap", "Small Cap"]
target_capitalization_bucket = str(target_gics_row.get("Capitalization", ""))
direct_peer_market_cap_frame = sector_historical_market_caps.copy()
if target_symbol not in direct_peer_market_cap_frame.columns and target_symbol in sector_pca_price_frame.columns:
    try:
        target_market_cap_series = fetch_historical_market_cap_series(
            target_symbol,
            cache_dir=market_cap_cache_dir,
            limit=market_cap_history_limit,
            from_date=market_cap_from_date,
            to_date=market_cap_to_date,
            refresh=refresh_market_cap_cache,
            requery_incomplete=requery_incomplete_market_cap_cache,
            request_pause_seconds=market_cap_request_pause_seconds,
        )
        if not target_market_cap_series.empty:
            direct_peer_market_cap_frame = pd.concat(
                [direct_peer_market_cap_frame, target_market_cap_series.rename(target_symbol)],
                axis=1,
            ).sort_index()
    except Exception as exc:
        print(f"Target market-cap history unavailable for {target_symbol}: {exc}")
direct_peer_200_day_returns_by_market_cap = {}
latest_direct_peer_200_day_return_rows = []

def format_market_cap(value):
    if pd.isna(value):
        return "Market Cap n/a"
    if abs(value) >= 1_000_000_000_000:
        return f"${value / 1_000_000_000_000:.2f}T"
    if abs(value) >= 1_000_000_000:
        return f"${value / 1_000_000_000:.1f}B"
    if abs(value) >= 1_000_000:
        return f"${value / 1_000_000:.1f}M"
    return f"${value:,.0f}"

for capitalization_bucket in direct_peer_capitalization_buckets:
    direct_peer_symbols = [
        symbol
        for symbol in sub_industry_peer_universe.loc[
            sub_industry_peer_universe["Capitalization"].astype(str).eq(capitalization_bucket),
            "Normalized Symbol",
        ].dropna().tolist()
        if symbol in sector_pca_price_frame.columns
    ]
    if capitalization_bucket == target_capitalization_bucket and target_symbol in sector_pca_price_frame.columns:
        direct_peer_symbols = [target_symbol, *direct_peer_symbols]
    direct_peer_symbols = list(dict.fromkeys(direct_peer_symbols))
    if not direct_peer_symbols:
        print(f"No {capitalization_bucket} direct Sub-Industry peer price histories are available to plot.")
        continue

    direct_peer_market_caps = (
        direct_peer_market_cap_frame
        .reindex(sector_pca_price_frame.index)
        .ffill()
        .reindex(columns=direct_peer_symbols)
        .loc[:rolling_return_plot_end_date]
    )
    direct_peer_latest_market_caps = direct_peer_market_caps.apply(
        lambda series: series.dropna().iloc[-1] if not series.dropna().empty else np.nan
    ).rename("Latest Market Cap")
    direct_peer_plot_order = direct_peer_latest_market_caps.sort_values(
        ascending=False,
        na_position="last",
    ).index.tolist()

    direct_peer_rolling_return_source = sector_pca_price_frame[direct_peer_plot_order].sort_index()
    direct_peer_return_frame = direct_peer_rolling_return_source.pct_change(
        return_horizon_days,
        fill_method=None,
    )
    direct_peer_return_frame = direct_peer_return_frame.loc[
        (direct_peer_return_frame.index >= rolling_return_plot_start_date)
        & (direct_peer_return_frame.index <= rolling_return_plot_end_date)
    ].dropna(how="all")
    if direct_peer_return_frame.empty:
        print(f"No {return_horizon_days}-day {capitalization_bucket} direct Sub-Industry peer returns are available to plot.")
        continue

    direct_peer_index_label = f"{capitalization_bucket} Sub-Industry Market-Cap Weighted Index"
    direct_peer_index_return_series = pd.Series(dtype="float64", name=direct_peer_index_label)
    bucket_index_frame = market_cap_weighted_index_frames.get(capitalization_bucket, pd.DataFrame())
    if "Sub-Industry" in bucket_index_frame.columns:
        direct_peer_index_return_series = bucket_index_frame["Sub-Industry"].pct_change(
            return_horizon_days,
            fill_method=None,
        ).rename(direct_peer_index_label)
        direct_peer_index_return_series = direct_peer_index_return_series.loc[
            (direct_peer_index_return_series.index >= rolling_return_plot_start_date)
            & (direct_peer_index_return_series.index <= rolling_return_plot_end_date)
        ].dropna()

    if direct_peer_index_return_series.empty:
        print(f"No {capitalization_bucket} Sub-Industry market-cap weighted index return series is available to plot.")
        direct_peer_200_day_returns_by_market_cap[capitalization_bucket] = direct_peer_return_frame
    else:
        direct_peer_200_day_returns_by_market_cap[capitalization_bucket] = pd.concat(
            [direct_peer_index_return_series, direct_peer_return_frame],
            axis=1,
        )

    direct_peer_return_fig = go.Figure()
    if not direct_peer_index_return_series.empty:
        direct_peer_return_fig.add_trace(
            go.Scatter(
                x=direct_peer_index_return_series.index,
                y=direct_peer_index_return_series,
                mode="lines",
                name=direct_peer_index_label,
                line=dict(color="#facc15", width=4.0, dash="dash"),
                hovertemplate=(
                    f"{direct_peer_index_label}<br>Role: Market-Cap Weighted Index<br>"
                    "%{x|%Y-%m-%d}<br>%{y:.2%}<extra></extra>"
                ),
            )
        )
    for symbol in direct_peer_plot_order:
        return_series = direct_peer_return_frame[symbol].dropna()
        if return_series.empty:
            continue
        company_role = "Target" if symbol == target_symbol else "Direct Peer"
        latest_market_cap = direct_peer_latest_market_caps.get(symbol, np.nan)
        market_cap_label = format_market_cap(latest_market_cap)
        trace_name = f"{symbol} | {company_role} | {market_cap_label}"
        direct_peer_return_fig.add_trace(
            go.Scatter(
                x=return_series.index,
                y=return_series,
                mode="lines",
                name=trace_name,
                line=dict(width=3.5 if symbol == target_symbol else 2.0),
                hovertemplate=(
                    f"{symbol}<br>Role: {company_role}<br>Capitalization: {capitalization_bucket}<br>"
                    f"Latest Market Cap: {market_cap_label}<br>"
                    "%{x|%Y-%m-%d}<br>%{y:.2%}<extra></extra>"
                ),
            )
        )
        latest_direct_peer_200_day_return_rows.append(
            {
                "Symbol": symbol,
                "Role": company_role,
                "Capitalization": capitalization_bucket,
                "Latest Market Cap": latest_market_cap,
                "Date": return_series.index[-1],
                f"{return_horizon_days}D Return": return_series.iloc[-1],
            }
        )

    direct_peer_return_fig.update_layout(
        title=f"{return_horizon_days}-Day Rolling Returns: {target_symbol} + {capitalization_bucket} Direct Sub-Industry Peers",
        template="plotly_dark",
        paper_bgcolor="#0b0f14",
        plot_bgcolor="#0b0f14",
        font=dict(color="#f8fafc"),
        xaxis=dict(title="Date", range=[rolling_return_plot_start_date, rolling_return_plot_end_date]),
        yaxis=dict(title="Rolling Return", tickformat=".0%", zeroline=True, zerolinecolor="#94a3b8"),
        legend=dict(title=f"{capitalization_bucket} Peers by Latest Market Cap"),
        hovermode="x unified",
        margin=dict(t=64, r=24, b=48, l=64),
        height=620,
    )
    direct_peer_return_fig.show()

if not sector_pca_200_day_returns_by_gics_level:
    raise ValueError("No market-cap weighted GICS level rolling returns are available. Run block 5 first.")

sector_pca_200_day_returns = pd.concat(sector_pca_200_day_returns_by_gics_level, axis=1)
latest_sector_pca_200_day_returns = pd.DataFrame(latest_sector_pca_200_day_return_rows)
latest_direct_peer_200_day_returns_by_market_cap = pd.DataFrame(latest_direct_peer_200_day_return_rows)

print(f"Computed {return_horizon_days}-trading-day rolling returns by GICS level.")
display(latest_sector_pca_200_day_returns.style.format({f"{return_horizon_days}D Return": "{:.2%}"}))
if not latest_direct_peer_200_day_returns_by_market_cap.empty:
    display(
        latest_direct_peer_200_day_returns_by_market_cap.style.format(
            {
                "Latest Market Cap": "${:,.0f}",
                f"{return_horizon_days}D Return": "{:.2%}",
            }
        )
    )
sector_pca_200_day_returns.tail(20)


In [ ]:
# 6B. Direct peer return-correlation heatmap
direct_peer_heatmap_years = 10
direct_peer_heatmap_min_observations = 60
direct_peer_heatmap_end_date = plot_end_date
direct_peer_heatmap_start_date = max(
    plot_start_date,
    direct_peer_heatmap_end_date - pd.DateOffset(years=direct_peer_heatmap_years),
)
direct_peer_heatmap_capitalization_buckets = ["Large Cap", "Mid Cap", "Small Cap"]
direct_peer_heatmap_target_capitalization = str(target_gics_row.get("Capitalization", ""))

direct_peer_heatmap_market_cap_frame = globals().get(
    "direct_peer_market_cap_frame",
    sector_historical_market_caps,
).copy()
if target_symbol not in direct_peer_heatmap_market_cap_frame.columns and target_symbol in sector_pca_price_frame.columns:
    try:
        target_market_cap_series = fetch_historical_market_cap_series(
            target_symbol,
            cache_dir=market_cap_cache_dir,
            limit=market_cap_history_limit,
            from_date=market_cap_from_date,
            to_date=market_cap_to_date,
            refresh=refresh_market_cap_cache,
            requery_incomplete=requery_incomplete_market_cap_cache,
            request_pause_seconds=market_cap_request_pause_seconds,
        )
        if not target_market_cap_series.empty:
            direct_peer_heatmap_market_cap_frame = pd.concat(
                [direct_peer_heatmap_market_cap_frame, target_market_cap_series.rename(target_symbol)],
                axis=1,
            ).sort_index()
    except Exception as exc:
        print(f"Target market-cap history unavailable for {target_symbol}: {exc}")

direct_peer_heatmap_symbol_rows = []
if target_symbol in sector_pca_price_frame.columns:
    direct_peer_heatmap_symbol_rows.append(
        {
            "Symbol": target_symbol,
            "Role": "Target",
            "Capitalization": direct_peer_heatmap_target_capitalization,
        }
    )

for capitalization_bucket in direct_peer_heatmap_capitalization_buckets:
    bucket_symbols = sub_industry_peer_universe.loc[
        sub_industry_peer_universe["Capitalization"].astype(str).eq(capitalization_bucket),
        "Normalized Symbol",
    ].dropna().tolist()
    for symbol in bucket_symbols:
        if symbol in sector_pca_price_frame.columns:
            direct_peer_heatmap_symbol_rows.append(
                {
                    "Symbol": symbol,
                    "Role": "Direct Peer",
                    "Capitalization": capitalization_bucket,
                }
            )

direct_peer_heatmap_order = pd.DataFrame(direct_peer_heatmap_symbol_rows).drop_duplicates("Symbol")
if direct_peer_heatmap_order.shape[0] < 2:
    raise ValueError("At least two direct Sub-Industry peer symbols with price history are required for a heatmap.")

direct_peer_heatmap_symbols = direct_peer_heatmap_order["Symbol"].tolist()
direct_peer_heatmap_latest_market_caps = (
    direct_peer_heatmap_market_cap_frame
    .reindex(sector_pca_price_frame.index)
    .ffill()
    .reindex(columns=direct_peer_heatmap_symbols)
    .loc[:direct_peer_heatmap_end_date]
    .apply(lambda series: series.dropna().iloc[-1] if not series.dropna().empty else np.nan)
    .rename("Latest Market Cap")
)

direct_peer_heatmap_order = direct_peer_heatmap_order.set_index("Symbol")
direct_peer_heatmap_order["Latest Market Cap"] = direct_peer_heatmap_latest_market_caps
capitalization_rank = {"Large Cap": 0, "Mid Cap": 1, "Small Cap": 2}
direct_peer_heatmap_order["Capitalization Rank"] = (
    direct_peer_heatmap_order["Capitalization"].map(capitalization_rank).fillna(99)
)
direct_peer_heatmap_order = direct_peer_heatmap_order.sort_values(
    ["Capitalization Rank", "Latest Market Cap"],
    ascending=[True, False],
    na_position="last",
)
ordered_direct_peer_heatmap_symbols = direct_peer_heatmap_order.index.tolist()

direct_peer_heatmap_price_frame = sector_pca_price_frame[ordered_direct_peer_heatmap_symbols].loc[
    (sector_pca_price_frame.index >= direct_peer_heatmap_start_date)
    & (sector_pca_price_frame.index <= direct_peer_heatmap_end_date)
].sort_index().ffill()
direct_peer_heatmap_returns = direct_peer_heatmap_price_frame.pct_change(fill_method=None).replace(
    [np.inf, -np.inf],
    np.nan,
).dropna(how="all")
valid_direct_peer_heatmap_symbols = [
    symbol
    for symbol in ordered_direct_peer_heatmap_symbols
    if direct_peer_heatmap_returns[symbol].dropna().shape[0] >= direct_peer_heatmap_min_observations
    and direct_peer_heatmap_returns[symbol].std(skipna=True) > 0
]
if len(valid_direct_peer_heatmap_symbols) < 2:
    raise ValueError("At least two direct peers need enough non-constant return history for a correlation heatmap.")

direct_peer_return_correlation = direct_peer_heatmap_returns[valid_direct_peer_heatmap_symbols].corr().loc[
    valid_direct_peer_heatmap_symbols,
    valid_direct_peer_heatmap_symbols,
]
direct_peer_correlation_heatmap_order = direct_peer_heatmap_order.loc[valid_direct_peer_heatmap_symbols].copy()
direct_peer_correlation_heatmap_order["Observations"] = direct_peer_heatmap_returns[
    valid_direct_peer_heatmap_symbols
].notna().sum()

def format_heatmap_market_cap(value):
    if pd.isna(value):
        return "n/a"
    if abs(value) >= 1_000_000_000_000:
        return f"${value / 1_000_000_000_000:.2f}T"
    if abs(value) >= 1_000_000_000:
        return f"${value / 1_000_000_000:.1f}B"
    if abs(value) >= 1_000_000:
        return f"${value / 1_000_000:.1f}M"
    return f"${value:,.0f}"

direct_peer_heatmap_labels = [
    f"{symbol}<br>{direct_peer_correlation_heatmap_order.at[symbol, 'Role']}<br>"
    f"{direct_peer_correlation_heatmap_order.at[symbol, 'Capitalization']} | "
    f"{format_heatmap_market_cap(direct_peer_correlation_heatmap_order.at[symbol, 'Latest Market Cap'])}"
    for symbol in valid_direct_peer_heatmap_symbols
]

direct_peer_heatmap_fig = go.Figure(
    data=go.Heatmap(
        z=direct_peer_return_correlation.values,
        x=direct_peer_heatmap_labels,
        y=direct_peer_heatmap_labels,
        zmin=-1,
        zmax=1,
        colorscale="RdBu",
        reversescale=True,
        colorbar=dict(title="Correlation"),
        hovertemplate="%{y}<br>%{x}<br>Correlation: %{z:.2f}<extra></extra>",
    )
)
direct_peer_heatmap_fig.update_layout(
    title=f"{target_symbol} Direct Sub-Industry Peer Daily Return Correlation Heatmap ({direct_peer_heatmap_years}Y)",
    template="plotly_dark",
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#f8fafc"),
    xaxis=dict(title="", tickangle=-45),
    yaxis=dict(title="", autorange="reversed"),
    margin=dict(t=80, r=32, b=180, l=180),
    height=max(700, 36 * len(valid_direct_peer_heatmap_symbols) + 260),
)
direct_peer_heatmap_fig.show()

display(
    direct_peer_correlation_heatmap_order
    .drop(columns=["Capitalization Rank"])
    .style.format({"Latest Market Cap": "${:,.0f}"})
)


In [ ]:
# 6C. GICS-level index return-correlation heatmap
gics_level_index_heatmap_years = 10
gics_level_index_heatmap_min_observations = 60
gics_level_index_heatmap_end_date = plot_end_date
gics_level_index_heatmap_start_date = max(
    plot_start_date,
    gics_level_index_heatmap_end_date - pd.DateOffset(years=gics_level_index_heatmap_years),
)
gics_level_index_heatmap_levels = ["Sector", "Industry Group", "Industry", "Sub-Industry"]

if "market_cap_weighted_index_frames" not in globals():
    raise ValueError("market_cap_weighted_index_frames is not available. Run block 5 first.")

gics_level_index_source = market_cap_weighted_index_frames.get("Total Cap", pd.DataFrame())
available_gics_level_index_columns = [
    gics_level
    for gics_level in gics_level_index_heatmap_levels
    if gics_level in gics_level_index_source.columns
]
if len(available_gics_level_index_columns) < 2:
    raise ValueError("At least two GICS-level market-cap weighted indexes are required for the heatmap.")

gics_level_index_price_frame = gics_level_index_source[available_gics_level_index_columns].loc[
    (gics_level_index_source.index >= gics_level_index_heatmap_start_date)
    & (gics_level_index_source.index <= gics_level_index_heatmap_end_date)
].sort_index().ffill()
gics_level_index_returns = gics_level_index_price_frame.pct_change(fill_method=None).replace(
    [np.inf, -np.inf],
    np.nan,
).dropna(how="all")
valid_gics_level_index_columns = [
    gics_level
    for gics_level in available_gics_level_index_columns
    if gics_level_index_returns[gics_level].dropna().shape[0] >= gics_level_index_heatmap_min_observations
    and gics_level_index_returns[gics_level].std(skipna=True) > 0
]
if len(valid_gics_level_index_columns) < 2:
    raise ValueError("At least two GICS-level indexes need enough non-constant return history for a heatmap.")

gics_level_index_return_correlation = gics_level_index_returns[valid_gics_level_index_columns].corr().loc[
    valid_gics_level_index_columns,
    valid_gics_level_index_columns,
]
gics_level_index_heatmap_labels = [
    f"{gics_level}<br>{target_gics_row[gics_level]}"
    for gics_level in valid_gics_level_index_columns
]

gics_level_index_heatmap_fig = go.Figure(
    data=go.Heatmap(
        z=gics_level_index_return_correlation.values,
        x=gics_level_index_heatmap_labels,
        y=gics_level_index_heatmap_labels,
        zmin=-1,
        zmax=1,
        colorscale="RdBu",
        reversescale=True,
        colorbar=dict(title="Correlation"),
        hovertemplate="%{y}<br>%{x}<br>Correlation: %{z:.2f}<extra></extra>",
    )
)
gics_level_index_heatmap_fig.update_layout(
    title=f"{target_symbol} Market-Cap Weighted GICS-Level Daily Return Correlation Heatmap ({gics_level_index_heatmap_years}Y)",
    template="plotly_dark",
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#f8fafc"),
    xaxis=dict(title="", tickangle=-30),
    yaxis=dict(title="", autorange="reversed"),
    margin=dict(t=90, r=32, b=130, l=160),
    height=620,
)
gics_level_index_heatmap_fig.show()

gics_level_index_heatmap_summary = pd.DataFrame(
    [
        {
            "GICS Level": gics_level,
            "GICS Name": target_gics_row[gics_level],
            "Index": f"Total Cap {gics_level} Market-Cap Weighted Index",
            "Observations": int(gics_level_index_returns[gics_level].notna().sum()),
            "Start Date": gics_level_index_returns[gics_level].dropna().index.min(),
            "End Date": gics_level_index_returns[gics_level].dropna().index.max(),
        }
        for gics_level in valid_gics_level_index_columns
    ]
)
display(gics_level_index_heatmap_summary)
gics_level_index_return_correlation


In [ ]:
# 7. Market-cap weighted index tracking diagnostics versus the sector ETF
tracking_oos_warmup_days = dynamic_pca_fit_window_days if "dynamic_pca_fit_window_days" in globals() else 252

try:
    from statsmodels.tsa.stattools import coint
except ImportError:
    coint = None

tracking_series = {}
if sector_benchmark_symbol and sector_benchmark_symbol in sector_pca_price_frame.columns:
    tracking_series[sector_benchmark_symbol] = normalize_price_series(
        sector_pca_price_frame[sector_benchmark_symbol], sector_benchmark_symbol
    )
if "market_cap_weighted_index_series" in globals():
    tracking_series.update(market_cap_weighted_index_series)
elif "sector_market_cap_weighted_index" in globals():
    tracking_series["Total Cap Sector"] = sector_market_cap_weighted_index

if globals().get("show_non_market_cap_indexes", False):
    synthetic_index_candidates = {
        "Static PCA Sector Index": "sector_pca_index",
        "Equal Weight Sector Index": "sector_equal_weight_index",
        "Volatility-Adjusted Equal Weight Sector Index": "sector_volatility_adjusted_equal_weight_index",
        "Large-Cap Static PCA Sector Index": "large_cap_sector_pca_index",
        "Quarterly Dynamic PCA Sector Index": "dynamic_sector_pca_index",
        "Monthly Dynamic PCA Sector Index": "monthly_dynamic_sector_pca_index",
        "Large-Cap Weekly Dynamic PCA Sector Index": "large_cap_weekly_dynamic_sector_pca_index",
        "Large-Cap Monthly Dynamic PCA Sector Index": "large_cap_monthly_dynamic_sector_pca_index",
    }
    for display_name, variable_name in synthetic_index_candidates.items():
        if variable_name in globals():
            tracking_series[display_name] = globals()[variable_name]

if sector_benchmark_symbol not in tracking_series:
    raise ValueError(f"Benchmark {sector_benchmark_symbol} is not available. Run the price and PCA cells first.")

tracking_source = pd.concat(tracking_series, axis=1).sort_index()
tracking_oos_start_date = (
    tracking_source.index[tracking_oos_warmup_days]
    if len(tracking_source) > tracking_oos_warmup_days
    else tracking_source.index.min()
)

def calculate_tracking_metrics(level_frame, benchmark_column, synthetic_column):
    pair_levels = level_frame[[benchmark_column, synthetic_column]].dropna()
    pair_levels = pair_levels.loc[pair_levels.index >= tracking_oos_start_date]
    if len(pair_levels) < 20:
        return None

    pair_returns = pair_levels.pct_change(fill_method=None).dropna()
    if len(pair_returns) < 20:
        return None

    benchmark_returns = pair_returns[benchmark_column]
    synthetic_returns = pair_returns[synthetic_column]
    active_returns = synthetic_returns - benchmark_returns

    benchmark_variance = float(np.var(benchmark_returns, ddof=0))
    beta = (
        float(np.cov(synthetic_returns, benchmark_returns, ddof=0)[0, 1] / benchmark_variance)
        if benchmark_variance > 0
        else np.nan
    )
    correlation = float(synthetic_returns.corr(benchmark_returns))
    normalized_levels = pair_levels / pair_levels.iloc[0]
    cumulative_drift = normalized_levels[synthetic_column] - normalized_levels[benchmark_column]

    coint_stat = np.nan
    coint_pvalue = np.nan
    if coint is not None and len(pair_levels) >= 30:
        coint_stat, coint_pvalue, _ = coint(pair_levels[benchmark_column], pair_levels[synthetic_column])

    return {
        "Synthetic Index": synthetic_column,
        "Start Date": pair_levels.index.min(),
        "End Date": pair_levels.index.max(),
        "Observations": len(pair_returns),
        "Tracking Error": float(active_returns.std(ddof=0) * np.sqrt(252)),
        "Cointegration p-value": float(coint_pvalue) if pd.notna(coint_pvalue) else np.nan,
        "Cointegrated 5%": bool(coint_pvalue < 0.05) if pd.notna(coint_pvalue) else np.nan,
        "R Squared": correlation ** 2 if pd.notna(correlation) else np.nan,
        "Beta": beta,
        "Correlation": correlation,
        "Maximum Cumulative Drift": float(cumulative_drift.abs().max()),
    }

tracking_metric_rows = [
    calculate_tracking_metrics(tracking_source, sector_benchmark_symbol, synthetic_column)
    for synthetic_column in tracking_source.columns
    if synthetic_column != sector_benchmark_symbol
]
synthetic_index_tracking_table = pd.DataFrame([row for row in tracking_metric_rows if row is not None])
if not synthetic_index_tracking_table.empty:
    synthetic_index_tracking_table = synthetic_index_tracking_table.sort_values("Tracking Error")

print(
    f"Out-of-sample diagnostics versus {sector_benchmark_symbol}: "
    f"starting {tracking_oos_start_date:%Y-%m-%d}."
)
display(
    synthetic_index_tracking_table.style.format(
        {
            "Tracking Error": "{:.2%}",
            "Cointegration p-value": "{:.4f}",
            "R Squared": "{:.2%}",
            "Beta": "{:.3f}",
            "Correlation": "{:.2%}",
            "Maximum Cumulative Drift": "{:.2%}",
        }
    )
)
